# AI Upscaler Backend
Opened by the Android app. Tap **Runtime ▸ Run all**, then **Allow** on the Google prompt, then return to the app. Keep this tab open.


In [ ]:
#@title Step 0 - Allow Drive access (tap Allow)
from google.colab import auth
auth.authenticate_user()

In [ ]:
#@title Step 1 - Setup (original Cell 1; HF token moved to Colab Secrets)
#@title **Cell 1 - Setup** { display-mode: "form" }
use_huggingface_cache = True

import os, sys, subprocess, shutil, time, hashlib

print("=" * 70)
print("Cell 1 : Setup — Proteus V3")
print("Cache source")
print("=" * 70)

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None
print("  HF_TOKEN from Colab Secrets" if HF_TOKEN else "  No HF_TOKEN secret set (anonymous)")

HF_DATASET_REPO = "legend2008/kzm-cache-v4"
HF_CACHE_PREFIX = "kzm_cache_v4"

LOCAL_PIP_CACHE = "/tmp/kzm_pip_cache"
LOCAL_MODELS    = "/content/models"

os.makedirs(LOCAL_PIP_CACHE, exist_ok=True)
os.makedirs(LOCAL_MODELS, exist_ok=True)

print("\n[0/6] Preparing HuggingFace Hub...")
try:
    import huggingface_hub
    print(f"  ✓ huggingface_hub {huggingface_hub.__version__} (system)")
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'], check=False)
    import huggingface_hub
    print(f"  ✓ huggingface_hub {huggingface_hub.__version__} (fresh)")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

from huggingface_hub import HfApi, hf_hub_download, snapshot_download
api = HfApi(token=HF_TOKEN)

if HF_TOKEN:
    try:
        me = api.whoami()
        print(f"  ✓ Authenticated as: {me['name']}  (read-only token)")
    except Exception as e:
        print(f"  ⚠ Token auth failed, falling back to anonymous: {e}")
        api = HfApi()
        HF_TOKEN = None
else:
    api = HfApi()

try:
    info = api.dataset_info(HF_DATASET_REPO)
    print(f"  ✓ Dataset accessible: {HF_DATASET_REPO}")
    print(f"    Private: {info.private}   Files: {len(info.siblings)}")
except Exception as e:
    raise RuntimeError(f"Cannot access dataset {HF_DATASET_REPO}: {e}")

print(f"  Dataset:  {HF_DATASET_REPO}")
print(f"  Token:    {'✓ authenticated (higher speed/limits)' if HF_TOKEN else '✗ anonymous (may be slow)'}")
print(f"  Cache →  {LOCAL_PIP_CACHE}")
print(f"  Models → {LOCAL_MODELS}")

print("\n[1/6] Restoring pip cache from HuggingFace Dataset (parallel)...")
print(f"  Downloading {HF_CACHE_PREFIX}/* → {LOCAL_PIP_CACHE} (parallel)...")
t_start = time.time()
try:
    snapshot_download(
        repo_id=HF_DATASET_REPO,
        repo_type="dataset",
        local_dir=LOCAL_PIP_CACHE,
        allow_patterns=[f"{HF_CACHE_PREFIX}/*", f"{HF_CACHE_PREFIX}/**/*"],
        token=HF_TOKEN,
        max_workers=8,
    )
    elapsed = time.time() - t_start
    print(f"  ✓ Snapshot download complete in {elapsed:.1f}s")
except Exception as e:
    elapsed = time.time() - t_start
    print(f"  ⚠ Snapshot download error after {elapsed:.1f}s: {e}")
    print(f"  → Falling back to file-by-file download (slower)...")
    try:
        repo_files = api.list_repo_files(HF_DATASET_REPO, repo_type="dataset")
    except Exception as e2:
        raise RuntimeError(f"Cannot list dataset files: {e2}")

    cache_files_in_repo = [f for f in repo_files if f.startswith(f"{HF_CACHE_PREFIX}/")]
    for i, repo_path in enumerate(cache_files_in_repo, 1):
        rel_path = repo_path[len(HF_CACHE_PREFIX) + 1:]
        local_path = os.path.join(LOCAL_PIP_CACHE, rel_path)
        os.makedirs(os.path.dirname(local_path), exist_ok=True)
        if os.path.exists(local_path) and os.path.getsize(local_path) > 0:
            continue
        try:
            downloaded = hf_hub_download(
                repo_id=HF_DATASET_REPO,
                repo_type="dataset",
                filename=repo_path,
                token=HF_TOKEN,
            )
            if os.path.abspath(downloaded) != os.path.abspath(local_path):
                shutil.copy2(downloaded, local_path)
            if i % 10 == 0:
                print(f"    ... {i}/{len(cache_files_in_repo)} files restored")
        except Exception as e3:
            print(f"    ⚠ Failed: {repo_path} — {e3}")

local_cache_files = []
for root, dirs, files in os.walk(os.path.join(LOCAL_PIP_CACHE, HF_CACHE_PREFIX)):
    for fname in files:
        local_cache_files.append(os.path.join(root, fname))

src_prefix = os.path.join(LOCAL_PIP_CACHE, HF_CACHE_PREFIX)
if os.path.isdir(src_prefix):
    for root, dirs, files in os.walk(src_prefix, topdown=False):
        rel = os.path.relpath(root, src_prefix)
        dst_dir = LOCAL_PIP_CACHE if rel == '.' else os.path.join(LOCAL_PIP_CACHE, rel)
        os.makedirs(dst_dir, exist_ok=True)
        for fname in files:
            src_file = os.path.join(root, fname)
            dst_file = os.path.join(dst_dir, fname)
            if os.path.abspath(src_file) != os.path.abspath(dst_file):
                try:
                    if os.path.exists(dst_file) and os.path.getsize(dst_file) == os.path.getsize(src_file):
                        continue
                    shutil.move(src_file, dst_file)
                except OSError:
                    shutil.copy2(src_file, dst_file)
    shutil.rmtree(src_prefix, ignore_errors=True)

total_local_files = sum(1 for dp, _, fns in os.walk(LOCAL_PIP_CACHE) for fn in fns)
total_local_bytes = sum(os.path.getsize(os.path.join(dp, fn)) for dp, _, fns in os.walk(LOCAL_PIP_CACHE) for fn in fns)
print(f"  ✓ Cache ready: {total_local_files} files, {total_local_bytes/1024/1024:.0f} MB in {LOCAL_PIP_CACHE}")

print("\n[2/6] Verifying ffmpeg...")
try:
    r = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
    if r.returncode != 0: raise RuntimeError("ffmpeg error")
    print(f"  OK {r.stdout.split(chr(10))[0]}")
except Exception:
    subprocess.run(['apt-get', 'update', '-qq'], check=False)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=False)

print("\n[3/6] Checking numpy...")
try:
    import numpy as np
    np_ver = np.__version__
except ImportError:
    np_ver = "missing"

if np_ver.startswith("1.26"):
    print(f"  numpy={np_ver} detected — restoring system numpy 2.x...")
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'numpy'], capture_output=True)
    _pip_cmd = [sys.executable, '-m', 'pip', 'install', 'numpy>=2.0', '--cache-dir', LOCAL_PIP_CACHE]
    subprocess.run(_pip_cmd, capture_output=True)
    for k in list(sys.modules.keys()):
        if k == 'numpy' or k.startswith('numpy.'):
            del sys.modules[k]
    import numpy as np

print(f"  ✓ numpy={np.__version__}")

print("\n[4/6] Installing TensorRT 10.x...")
def pip_install(*pkgs):
    cmd = [sys.executable, '-m', 'pip', 'install', '--prefer-binary', '--cache-dir', LOCAL_PIP_CACHE]
    cmd.extend(pkgs)
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(f"  ⚠ Cache install failed, retrying fresh download...")
        cmd_nocache = [sys.executable, '-m', 'pip', 'install']
        cmd_nocache.extend(pkgs)
        r = subprocess.run(cmd_nocache, capture_output=True, text=True)
        if r.returncode != 0:
            print(f"  ⚠ Install failed: {r.stderr[-300:]}")
    return r.returncode == 0

subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'tensorrt', 'tensorrt-cu12', 'tensorrt-cu12-bindings', 'tensorrt-cu12-libs'], capture_output=True)

pip_install('tensorrt-cu12>=10.0,<11')
pip_install('tensorrt>=10.0,<11')

for k in list(sys.modules.keys()):
    if 'tensorrt' in k:
        del sys.modules[k]

import tensorrt as trt
print(f"  ✓ TensorRT {trt.__version__}")

print("  Installing pycuda...")
pip_install('pycuda')

try:
    import cv2
    print(f"  ✓ cv2={cv2.__version__} (system)")
except ImportError:
    pip_install('opencv-python-headless')
    import cv2
    print(f"  ✓ cv2={cv2.__version__} (fresh)")

print(f"  ✓ numpy={np.__version__}  TRT={trt.__version__}  cv2={cv2.__version__}")

print("\n  Checking GPU...")
dev = None
try:
    import pycuda.driver as cuda
    cuda.init()
    dev = cuda.Device(0)
    ctx = dev.make_context()
    try:
        free_mem, total_mem = cuda.mem_get_info()
        cc_major, cc_minor = dev.compute_capability()
        print(f"  GPU: {dev.name()}")
        print(f"  Memory: {free_mem/1024**3:.1f}GB free / {total_mem/1024**3:.1f}GB total")
        print(f"  Compute Capability: {cc_major}.{cc_minor}")
        if cc_major >= 6:
            print(f"  ✓ FP16 supported")
        else:
            raise RuntimeError(f"FP16 needs CC>=6.0, got {cc_major}.{cc_minor}")
    finally:
        ctx.pop()
        ctx.detach()
except Exception as e:
    print(f"  ⚠ GPU: {e}")

print("\n[5/6] Preparing TRT engines (FP16 + FP32, 1x + 2x)...")
DST = LOCAL_MODELS

for _f in os.listdir(DST):
    if _f.endswith('.trt'):
        _old = os.path.join(DST, _f)
        try:
            if os.path.getsize(_old) < 1000:
                os.remove(_old)
        except: pass

ENGINES = {
    'prob-v3-fgnet-fp32-576x672-1x.trt': 'https://huggingface.co/buckets/legend2008/proteus-v3-backup/resolve/models/prob-v3-fgnet-fp32-576x672-1x.trt?download=true',
    'prob-v3-fgnet-fp32-576x672-2x.trt': 'https://huggingface.co/buckets/legend2008/proteus-v3-backup/resolve/models/prob-v3-fgnet-fp32-576x672-2x.trt?download=true',
    'prob-v3-fgnet-fp16-576x672-1x.trt': 'https://huggingface.co/buckets/legend2008/proteus-v3-backup/resolve/models/prob-v3-fgnet-fp16-576x672-1x.trt?download=true',
    'prob-v3-fgnet-fp16-576x672-2x.trt': 'https://huggingface.co/buckets/legend2008/proteus-v3-backup/resolve/models/prob-v3-fgnet-fp16-576x672-2x.trt?download=true',
}

for fname, url in ENGINES.items():
    local_path = os.path.join(DST, fname)
    if os.path.exists(local_path) and os.path.getsize(local_path) > 1_000_000:
        print(f"  ✓ {fname} (already local, {os.path.getsize(local_path)//1048576} MB)")
        continue

    print(f"  ⬇ Downloading {fname}...")
    r_dl = subprocess.run(['curl', '-sL', '--retry', '3', '--connect-timeout', '30', '-o', local_path, url], capture_output=True, text=True)
    if r_dl.returncode != 0 or not os.path.exists(local_path) or os.path.getsize(local_path) < 1000:
        print(f"  ✗ Download failed!")
        if r_dl.stderr:
            print(f"    Error: {r_dl.stderr[-200:]}")
        continue
    print(f"    ✓ Downloaded ({os.path.getsize(local_path)//1048576} MB)")

print("\n[6/6] Verifying TensorRT engines...")
TRT_LOGGER = trt.Logger(trt.Logger.INFO)
runtime = trt.Runtime(TRT_LOGGER)

for fname in sorted(ENGINES.keys()):
    p = os.path.join(DST, fname)
    if not os.path.exists(p) or os.path.getsize(p) < 1000:
        print(f"  ✗ {fname}: MISSING")
        continue
    with open(p, 'rb') as f:
        engine = runtime.deserialize_cuda_engine(f.read())
    if engine:
        ctx = engine.create_execution_context()
        print(f"  ✓ {fname}: {engine.num_io_tensors} tensors, {os.path.getsize(p)//1048576} MB")
        for i in range(engine.num_io_tensors):
            name = engine.get_tensor_name(i)
            mode = engine.get_tensor_mode(name)
            shape = engine.get_tensor_shape(name)
            dtype = engine.get_tensor_dtype(name)
            ms = "IN " if mode == trt.TensorIOMode.INPUT else "OUT"
            dtype_str = "FP16" if dtype == trt.DataType.HALF else ("FP32" if dtype == trt.DataType.FLOAT else str(dtype))
            print(f"    [{ms}] {name}: shape={shape}, dtype={dtype_str}")
        del ctx, engine
    else:
        print(f"  ✗ {fname}: failed to deserialize")

print(f"\n{'='*70}")
print(f"Summary")
print(f"{'='*70}")
print(f"  TensorRT:  {trt.__version__}")
print(f"  numpy:     {np.__version__}")
print(f"  cv2:       {cv2.__version__}")
print(f"  GPU:       {dev.name() if dev else 'N/A'}")
print(f"\n  Cache source: HuggingFace Dataset '{HF_DATASET_REPO}'")
print(f"  Local pip cache: {LOCAL_PIP_CACHE}  ({sum(os.path.getsize(os.path.join(dp,fn)) for dp,_,fns in os.walk(LOCAL_PIP_CACHE) for fn in fns)/1048576:.0f} MB)")
print(f"\n  Engine files in {DST}/:")
for f in sorted(os.listdir(DST)):
    fp = os.path.join(DST, f)
    if os.path.isfile(fp) and f.endswith('.trt'):
        size = os.path.getsize(fp) / 1048576
        print(f"    {f}  ({size:.1f} MB)")

print(f"\n{'='*70}")
print("[OK] Cell 1 complete!")
print("=" * 70)

In [ ]:
%%writefile /content/cell2_original.py
# @title **Cell 2** - Proteus V3 ( **Proteus V3 Premium Upscaler "Private"** ) { display-mode: "form" }
import os, time, math, subprocess, shutil, json, threading, sys
import numpy as np
import cv2

# ═══════════════════════════════════════════════════════════════════════════
# Optional imports (TRT/pycuda only needed for GPU mode)
# ═══════════════════════════════════════════════════════════════════════════
try:
    import tensorrt as trt
    _HAS_TRT = True
except ImportError:
    trt = None
    _HAS_TRT = False

try:
    import pycuda.driver as cuda
    import pycuda.autoinit
    _HAS_PYCUDA = True
except ImportError:
    cuda = None
    _HAS_PYCUDA = False

# ═══════════════════════════════════════════════════════════════════════════
# GUI INPUTS
# ═══════════════════════════════════════════════════════════════════════════

#@markdown ---
#@markdown ### 📁 **File & Scaling**
#@markdown Path to your input video or image file.
INPUT_PATH = "" #@param {type:"string"}
#@markdown Upscaling factor (2x = High resolution, 1x = No size change).
SCALE = "2x" #@param ["1x", "2x"]
SCALE = int(SCALE.replace("x", ""))  # Convert to int

#@markdown ---
#@markdown ### 🎨 **Quality Presets & Engine**
#@markdown Select `Custom` to manually adjust the sliders below. Presets will override sliders.
Quality_Preset = "Custom" #@param ["Custom", "V1 Anime 1 Pass", "V2 Anime 2 Pass", "V3 Anime High Quality 2 Pass"]
#@markdown `ON` = FP16 (Faster, slightly less precise) | `OFF` = FP32 (Slower, max precision)
Upscaling_Speed_Up = "ON" #@param ["ON", "OFF"]

#@markdown ---
#@markdown ### 🎛️ **Custom Sliders** (Only active if Preset is `Custom`)
#@markdown Anti-Alias & Deblur (-100 to +100). Clears blur and jagged edges.
Anti_Alias_Deblur = 50 #@param {type:"slider", min:-100, max:100, step:1}
#@markdown Reduce Noise (0 to 100). Removes grain and artifacts.
ReduceNoise = 17 #@param {type:"slider", min:0, max:100, step:1}
#@markdown Recover Details (0 to 100). Brings back lost fine details.
RecoverDetails = 100 #@param {type:"slider", min:0, max:100, step:1}
#@markdown Dehalo (0 to 100). Removes white outlines around edges.
Dehalo = 10 #@param {type:"slider", min:0, max:100, step:1}
#@markdown Sharpen (0 to 100). Enhances edge sharpness.
Sharpen = 20 #@param {type:"slider", min:0, max:100, step:1}
#@markdown Revert Compression (0 to 100). Fixes compression artifacts.
RevertCompression = 100 #@param {type:"slider", min:0, max:100, step:1}
#@markdown Recover Original Details (0 to 100). Blends original high-freq details back.
Recover_Original_Details = 0 #@param {type:"slider", min:0, max:100, step:1}

#@markdown ---
#@markdown ### 📐 **Resolutions**
#@markdown Pre-resize Video before AI upscaling (`No Resize` keeps original).
Input_Video_Resolution = "1080p" #@param ["1080p", "1440p", "2160p", "No Resize"]
#@markdown Pre-resize Image before AI upscaling.
Input_Image_Resolution = "1440p" #@param ["1080p", "1440p", "2160p", "No Resize"]
#@markdown Post-resize Video after AI upscaling (`Original AI Output` keeps AI size).
Output_Video_Resize = "1440p 2K" #@param ["Original AI Output", "1080p HD", "1440p 2K", "2160p 4K"]

#@markdown ---
#@markdown ### 🖼️ **Image Output Settings**
#@markdown Output format for image files.
Image_Format = "PNG 8bit" #@param ["PNG 8bit", "PNG 16bit", "TIFF 10bit", "TIFF 12bit", "TIFF 16bit"]

#@markdown ---
#@markdown ### 🎬 **Video Output Settings**
#@markdown Video encoder format. `ProRes` and `FFV1` are high-bit-depth/lossless.
Encoder = "x265 8bit" #@param ["x264 8bit", "x265 8bit", "x265 10bit", "ProRes 4444 12bit", "FFV1 16bit Lossless"]
#@markdown Save every frame as 16-bit PNG (`OFF` = output as a single video file).
Save_Frames = "OFF" #@param ["ON", "OFF"]
#@markdown Video Quality (18=High Quality, 51=Low Quality, 13=Lossless).
Quality = 16 #@param {type:"slider", min:0, max:51, step:1}

#@markdown ---
#@markdown ### ⚙️ **System Settings**
#@markdown Automatically download the output file when finished.
Auto_Download = "ON" #@param ["ON", "OFF"]
Verbose_Log = "OFF"

# ═══════════════════════════════════════════════════════════════════════════
# Constants
# ═══════════════════════════════════════════════════════════════════════════
IMAGE_PASSES = 3       # hardcoded, no slider
VIDEO_FIRST_PASSES = 3
PREFLIGHT = 10           # hardcoded, no slider
TILE_H, TILE_W = 576, 672
OVERLAP = 192
CHANNELS = {1: 15, 2: 24}  # NO 4x (was 60)
SWS_FLAGS = 'spline+accurate_rnd+full_chroma_int'
COLOR_TRC, COLORSPACE, COLOR_PRIMARIES = '1', '5', '6'
MOVFLAGS = 'frag_keyframe+empty_moov+delay_moov+use_metadata_tags+write_colr'
USE_TEMPORAL = True  # always ON

_INPUT_RES_MAP = {"1080p": 1080, "1440p": 1440, "2160p": 2160, "No Resize": 99999}
MAX_INPUT_HEIGHT = _INPUT_RES_MAP.get(Input_Video_Resolution, 1080)
MAX_IMAGE_INPUT_DIM = _INPUT_RES_MAP.get(Input_Image_Resolution, 2160)
_OUTPUT_RES_MAP = {"Original AI Output": 0, "1080p HD": 1080, "1440p 2K": 1440, "2160p 4K": 2160}
OUTPUT_TARGET_SHORT_SIDE = _OUTPUT_RES_MAP.get(Output_Video_Resize, 0)
IS_IMAGE_INPUT = any(INPUT_PATH.lower().endswith(e) for e in ['.jpg','.png','.jpeg','.bmp','.webp','.tiff'])

# ═══════════════════════════════════════════════════════════════════════════
# Quality Presets (auto-apply when Quality_Preset != "Custom")
# ═══════════════════════════════════════════════════════════════════════════
QUALITY_PRESETS = {
    "V1 Anime 1 Pass": {
        "scale": 2,
        "anti_alias_deblur": 100,
        "reduce_noise": 100,
        "recover_details": 100,
        "dehalo": 50,
        "sharpen": 35,
        "revert_compression": 100,
        "recover_original": 0,
        "passes": 1,
    },
    "V2 Anime 2 Pass": {
        "scale": 2,
        "pass1": {"anti_alias_deblur": -50, "reduce_noise": 100, "recover_details": 79,
                  "dehalo": 65, "sharpen": 5, "revert_compression": 100, "recover_original": 0},
        "pass2": {"anti_alias_deblur": 80, "reduce_noise": 100, "recover_details": 87,
                  "dehalo": 70, "sharpen": 20, "revert_compression": 100, "recover_original": 0},
        "passes": 2,
    },
    "V3 Anime High Quality 2 Pass": {
        "scale": 2,
        "pass1": {"anti_alias_deblur": 100, "reduce_noise": 100, "recover_details": 100,
                  "dehalo": 45, "sharpen": 20, "revert_compression": 100, "recover_original": 0},
        "pass2": {"anti_alias_deblur": 100, "reduce_noise": 100, "recover_details": 100,
                  "dehalo": 50, "sharpen": 23, "revert_compression": 100, "recover_original": 0},
        "passes": 2,
    },
}

# Apply preset (auto-apply when not Custom — no Apply_Preset toggle)
if Quality_Preset != "Custom" and Quality_Preset in QUALITY_PRESETS:
    _preset = QUALITY_PRESETS[Quality_Preset]
    SCALE = _preset["scale"]
    PASS_COUNT = _preset["passes"]
    if PASS_COUNT == 1:
        PASS1_SLIDERS = {
            "anti_alias_deblur": _preset["anti_alias_deblur"],
            "reduce_noise": _preset["reduce_noise"],
            "recover_details": _preset["recover_details"],
            "dehalo": _preset["dehalo"],
            "sharpen": _preset["sharpen"],
            "revert_compression": _preset["revert_compression"],
            "recover_original": _preset["recover_original"],
        }
        PASS2_SLIDERS = None
    else:
        PASS1_SLIDERS = _preset["pass1"]
        PASS2_SLIDERS = _preset["pass2"]
else:
    # Custom mode — use GUI slider values directly
    PASS_COUNT = 1
    PASS1_SLIDERS = {
        "anti_alias_deblur": Anti_Alias_Deblur,
        "reduce_noise": ReduceNoise,
        "recover_details": RecoverDetails,
        "dehalo": Dehalo,
        "sharpen": Sharpen,
        "revert_compression": RevertCompression,
        "recover_original": Recover_Original_Details,
    }
    PASS2_SLIDERS = None

INJECT_OVERLAY = {}            # global, mutated by set_pass_params
preBlur_val = 0.0              # legacy alias for INJECT_OVERLAY[3]
noise_val = 0.0                # legacy alias for INJECT_OVERLAY[4]
details_val = 0.0              # legacy alias for INJECT_OVERLAY[5]
halo_val = 0.0                 # legacy alias for INJECT_OVERLAY[6]
sharpen_val = 0.0              # legacy alias for INJECT_OVERLAY[7]
compression_val = 0.0          # legacy alias for INJECT_OVERLAY[8]
CURRENT_RECOVER_ORIGINAL = 0   # blend 0-100 for cv2-based detail recovery

def set_pass_params(slider_dict):
    """Update all global slider params for the current pass."""
    global INJECT_OVERLAY, preBlur_val, noise_val, details_val, halo_val, sharpen_val, compression_val
    global CURRENT_RECOVER_ORIGINAL
    INJECT_OVERLAY = {
        3: float(slider_dict["anti_alias_deblur"]) / 100.0,   # signed: -1 to +1
        4: float(slider_dict["reduce_noise"]) / 100.0,
        5: float(slider_dict["recover_details"]) / 100.0,
        6: float(slider_dict["dehalo"]) / 100.0,
        7: float(slider_dict["sharpen"]) / 100.0,
        8: float(slider_dict["revert_compression"]) / 100.0,
    }
    preBlur_val     = INJECT_OVERLAY[3]
    noise_val       = INJECT_OVERLAY[4]
    details_val     = INJECT_OVERLAY[5]
    halo_val        = INJECT_OVERLAY[6]
    sharpen_val     = INJECT_OVERLAY[7]
    compression_val = INJECT_OVERLAY[8]
    CURRENT_RECOVER_ORIGINAL = int(slider_dict.get("recover_original", 0))

# Initial pass setup (Pass 1)
set_pass_params(PASS1_SLIDERS)

# Determine precision: FP16 or FP32
_USE_FP16 = (Upscaling_Speed_Up == "ON")
_PRECISION_TAG = "fp16" if _USE_FP16 else "fp32"

if Verbose_Log == "ON":
    print("=" * 70)
    print(f"Proteus V3 — TensorRT ({_PRECISION_TAG.upper()}) Pipeline")
    print("=" * 70)
    print(f"  Input:  {INPUT_PATH}")
    print(f"  Scale:  {SCALE}x")
    print(f"  Type:   {'Image' if IS_IMAGE_INPUT else 'Video'}")
    print(f"  Quality Preset: {Quality_Preset}  (passes: {PASS_COUNT})")
    print(f"  Image Passes: {IMAGE_PASSES}  (hardcoded)")
    print(f"  Preflight Warmup: {PREFLIGHT} frames  (hardcoded)")
    print(f"  Temporal Processing: ON (always)")

    print(f"\n  🎛️  Slider → Channel Mapping (Pass 1):")
    print(f"    Anti_Alias_Deblur = {PASS1_SLIDERS['anti_alias_deblur']:>4d}  → ch3 (signed, range -1 to +1)")
    print(f"    ReduceNoise       = {PASS1_SLIDERS['reduce_noise']:>4d}  → ch4")
    print(f"    RecoverDetails    = {PASS1_SLIDERS['recover_details']:>4d}  → ch5")
    print(f"    Dehalo            = {PASS1_SLIDERS['dehalo']:>4d}  → ch6")
    print(f"    Sharpen           = {PASS1_SLIDERS['sharpen']:>4d}  → ch7")
    print(f"    RevertCompression = {PASS1_SLIDERS['revert_compression']:>4d}  → ch8")
    print(f"    RecoverOriginal   = {PASS1_SLIDERS['recover_original']:>4d}  → cv2 post-process blend")

    if PASS2_SLIDERS is not None:
        print(f"\n  🎛️  Slider → Channel Mapping (Pass 2):")
        print(f"    Anti_Alias_Deblur = {PASS2_SLIDERS['anti_alias_deblur']:>4d}  → ch3 (signed)")
        print(f"    ReduceNoise       = {PASS2_SLIDERS['reduce_noise']:>4d}  → ch4")
        print(f"    RecoverDetails    = {PASS2_SLIDERS['recover_details']:>4d}  → ch5")
        print(f"    Dehalo            = {PASS2_SLIDERS['dehalo']:>4d}  → ch6")
        print(f"    Sharpen           = {PASS2_SLIDERS['sharpen']:>4d}  → ch7")
        print(f"    RevertCompression = {PASS2_SLIDERS['revert_compression']:>4d}  → ch8")
        print(f"    RecoverOriginal   = {PASS2_SLIDERS['recover_original']:>4d}  → cv2 post-process blend")

    print(f"\n  📋 INJECT_OVERLAY (Pass 1 channels overridden):")
    for ch_idx in sorted(INJECT_OVERLAY.keys()):
        if ch_idx in (0, 1, 2):              group = "lr_curr RGB"
        elif ch_idx in (3, 4, 5, 6, 7, 8):   group = "params (slider)"
        elif ch_idx in (9, 10, 11):          group = "lr_prev RGB"
        else:                                 group = "packed_hr_prev"
        print(f"    ch[{ch_idx:2d}] ({group:<18}) = {INJECT_OVERLAY[ch_idx]:+.3f}")

    if not IS_IMAGE_INPUT:
        print(f"\n  Encoder: {Encoder}, Quality: {Quality}")
        print(f"  Save Frames: {Save_Frames}")
        print(f"  Input Video Resolution: {Input_Video_Resolution}")
        print(f"  Output Video Resize: {Output_Video_Resize}")
    else:
        print(f"\n  Image Format: {Image_Format}")
        print(f"  Input Image Resolution: {Input_Image_Resolution}")

    print(f"\n  ★ Architecture: lr_curr(3) + params(6) + lr_prev(3) + packed_hr_prev(12) = 24ch")
    print(f"  ★ Multi-pass: {IMAGE_PASSES} passes for images, {VIDEO_FIRST_PASSES} for video first-frame.")
    print(f"  ★ 2-pass video mode: {'ON (V2/V3 preset)' if PASS_COUNT == 2 else 'OFF (single pass)'}")
else:
    print("=" * 70)
    print(f"  Input:  {INPUT_PATH}")
    print(f"  Scale:  {SCALE}x")
    print(f"  Type:   {'Image' if IS_IMAGE_INPUT else 'Video'}")
    print(f"  Quality Preset: {Quality_Preset}")
    if not IS_IMAGE_INPUT:
        print(f"  Encoder: {Encoder}, Quality: {Quality}")
        print(f"  Save Frames: {Save_Frames}")
        print(f"  Input Video Resolution: {Input_Video_Resolution}")
        print(f"  Output Video Resize: {Output_Video_Resize}")
    else:
        print(f"  Image Format: {Image_Format}")
        print(f"  Input Image Resolution: {Input_Image_Resolution}")

# ═══════════════════════════════════════════════════════════════════════════
# Load Model (TensorRT or ONNX fallback)
# ═══════════════════════════════════════════════════════════════════════════
MODEL_PATH = None
MODEL_IS_ONNX = False
_USE_ORT = False

# Candidate paths: try requested precision TRT first, then other, then ONNX
_CANDIDATE_PATHS = [
    f'/content/models/prob-v3-fgnet-{_PRECISION_TAG}-576x672-{SCALE}x.trt',
    f'/content/prob-v3-fgnet-{_PRECISION_TAG}-576x672-{SCALE}x.trt',
    # Fallback: other precision
    f'/content/models/prob-v3-fgnet-{"fp32" if _USE_FP16 else "fp16"}-576x672-{SCALE}x.trt',
    f'/content/prob-v3-fgnet-{"fp32" if _USE_FP16 else "fp16"}-576x672-{SCALE}x.trt',
    # Last resort: ONNX
    f'/content/models/prob-v3-fgnet-fp32-576x672-{SCALE}x.onnx',
    f'/content/prob-v3-fgnet-fp32-576x672-{SCALE}x.onnx',
]
for _p in _CANDIDATE_PATHS:
    if os.path.exists(_p):
        MODEL_PATH = _p
        MODEL_IS_ONNX = _p.endswith('.onnx')
        break

if MODEL_PATH is None:
    print(f"\n  Model not found for SCALE={SCALE}x. Searched:")
    for _p in _CANDIDATE_PATHS:
        print(f"    {_p}")
    raise FileNotFoundError(
        f"Model file not found for SCALE={SCALE}x. "
        f"Use SCALE=2 (recommended)."
    )

import torch as _torch_check
_HAS_CUDA = _torch_check.cuda.is_available()
if not _HAS_CUDA and not MODEL_IS_ONNX:
    print(f"\n  No CUDA GPU detected. Trying ONNX fallback...")
    for _p in _CANDIDATE_PATHS:
        if _p.endswith('.onnx') and os.path.exists(_p):
            MODEL_PATH = _p
            MODEL_IS_ONNX = True
            break

if Verbose_Log == "ON":
    print(f"\n  Loading model: {MODEL_PATH}\n  Precision: {_PRECISION_TAG.upper()} (Upscaling Speed Up: {Upscaling_Speed_Up})")
else:
    print("Loading model: Model Loaded")

if MODEL_IS_ONNX:
    if Verbose_Log == "ON":
        print(f"  Mode: ONNX Runtime ({'GPU' if _HAS_CUDA else 'CPU'})")
    import onnxruntime as _ort
    _providers = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if _HAS_CUDA else ['CPUExecutionProvider']
    ort_sess = _ort.InferenceSession(MODEL_PATH, providers=_providers)
    _input_meta = ort_sess.get_inputs()[0]
    _output_meta = ort_sess.get_outputs()[0]
    input_name = _input_meta.name
    output_name = _output_meta.name
    input_shape = tuple(_input_meta.shape)
    output_shape = tuple(_output_meta.shape)
    TILE_H, TILE_W = input_shape[1], input_shape[2]
    if Verbose_Log == "ON":
        print(f"  Input:  {input_name} shape={input_shape}")
        print(f"  Output: {output_name} shape={output_shape}")
        print(f"  Tile size: {TILE_H}x{TILE_W}")
    _USE_ORT = True
    d_input = None
    d_output = None
    stream = None
    context = None
    engine = None
else:
    if Verbose_Log == "ON":
        print(f"  Mode: TensorRT")
    TRT_LOGGER = trt.Logger(trt.Logger.INFO)
    runtime = trt.Runtime(TRT_LOGGER)
    with open(MODEL_PATH, 'rb') as f:
        engine = runtime.deserialize_cuda_engine(f.read())

    if engine is None:
        if Verbose_Log == "ON":
            print(f"\n  TRT engine failed to deserialize! Trying ONNX fallback...")
        for _p in _CANDIDATE_PATHS:
            if _p.endswith('.onnx') and os.path.exists(_p):
                MODEL_PATH = _p
                MODEL_IS_ONNX = True
                break
        if MODEL_IS_ONNX:
            import onnxruntime as _ort
            _providers = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if _HAS_CUDA else ['CPUExecutionProvider']
            ort_sess = _ort.InferenceSession(MODEL_PATH, providers=_providers)
            _input_meta = ort_sess.get_inputs()[0]
            _output_meta = ort_sess.get_outputs()[0]
            input_name = _input_meta.name
            output_name = _output_meta.name
            input_shape = tuple(_input_meta.shape)
            output_shape = tuple(_output_meta.shape)
            TILE_H, TILE_W = input_shape[1], input_shape[2]
            if Verbose_Log == "ON":
                print(f"  ONNX fallback loaded. Tile size: {TILE_H}x{TILE_W}")
            _USE_ORT = True
            d_input = None
            d_output = None
            stream = None
            context = None
            engine = None
        else:
            raise RuntimeError(
                "TRT engine failed AND no ONNX model found. "
                "Upload .tz file or use GPU runtime."
            )
    else:
        context = engine.create_execution_context()
        input_name = None
        output_name = None
        input_shape = None
        output_shape = None
        _io_dtype = None
        for i in range(engine.num_io_tensors):
            name = engine.get_tensor_name(i)
            mode = engine.get_tensor_mode(name)
            shape = engine.get_tensor_shape(name)
            dtype = engine.get_tensor_dtype(name)
            if mode == trt.TensorIOMode.INPUT:
                input_name = name
                input_shape = tuple(shape)
                _io_dtype = dtype
                dtype_str = "FP16" if dtype == trt.DataType.HALF else "FP32"
                if Verbose_Log == "ON":
                    print(f"  Input:  {name} shape={shape} dtype={dtype_str}")
            else:
                output_name = name
                output_shape = tuple(shape)
                dtype_str = "FP16" if dtype == trt.DataType.HALF else "FP32"
                if Verbose_Log == "ON":
                    print(f"  Output: {name} shape={shape} dtype={dtype_str}")
        TILE_H, TILE_W = input_shape[1], input_shape[2]

        # Detect actual scale from input channels (15=1x, 24=2x)
        _actual_ch = input_shape[-1]
        if _actual_ch == 15:
            _detected_scale = 1
        elif _actual_ch == 24:
            _detected_scale = 2
        else:
            _detected_scale = SCALE  # fallback to GUI

        if _detected_scale != SCALE and Verbose_Log == "ON":
            print(f"  ⚠ WARNING: Engine is {_detected_scale}x but GUI SCALE={SCALE}x!")
            print(f"  → Auto-correcting SCALE to {_detected_scale}x (from engine)")
        SCALE = _detected_scale

        # Determine numpy dtype for I/O based on engine tensor dtype
        _NP_IO_DTYPE = np.float16 if _io_dtype == trt.DataType.HALF else np.float32
        _IO_ITEMSIZE = np.dtype(_NP_IO_DTYPE).itemsize
        if Verbose_Log == "ON":
            print(f"  I/O dtype: {'FP16 (2 bytes)' if _NP_IO_DTYPE == np.float16 else 'FP32 (4 bytes)'}")

        input_size = trt.volume(input_shape) * _IO_ITEMSIZE
        output_size = trt.volume(output_shape) * _IO_ITEMSIZE
        d_input = cuda.mem_alloc(input_size)
        d_output = cuda.mem_alloc(output_size)
        stream = cuda.Stream()
        context.set_tensor_address(input_name, int(d_input))
        context.set_tensor_address(output_name, int(d_output))

        _engine_size_mb = os.path.getsize(MODEL_PATH) / 1048576
        _has_fp16 = any(engine.get_tensor_dtype(engine.get_tensor_name(i)) == trt.DataType.HALF
                         for i in range(engine.num_io_tensors))
        _engine_prec = "FP16" if _has_fp16 or "fp16" in MODEL_PATH else "FP32"
        if Verbose_Log == "ON":
            print(f"  TRT engine loaded. Tile size: {TILE_H}x{TILE_W}, Scale: {SCALE}x")
            print(f"  Engine file: {os.path.basename(MODEL_PATH)} ({_engine_size_mb:.1f} MB)")
            print(f"  Engine precision: {_engine_prec} (I/O dtype: {engine.get_tensor_dtype(engine.get_tensor_name(0))})")

# ═══════════════════════════════════════════════════════════════════════════
# Helper Functions
# ═══════════════════════════════════════════════════════════════════════════
def make_feather_mask(h, w, overlap):
    m = np.ones((h, w), dtype=np.float32)
    if overlap > 0:
        r = np.linspace(0, 1, overlap, dtype=np.float32)
        m[:, :overlap] *= r[np.newaxis, :]
        m[:, -overlap:] *= r[::-1][np.newaxis, :]
        m[:overlap, :] *= r[:, np.newaxis]
        m[-overlap:, :] *= r[::-1][:, np.newaxis]
    return m

def reflect_pad(img, th, tw):
    h, w = img.shape[:2]
    ph, pw = max(0, th - h), max(0, tw - w)
    if ph > 0 or pw > 0:
        return np.pad(img, ((0, ph), (0, pw), (0, 0)), mode='reflect')
    return img

def compute_tile_positions(ts, tile, overlap):
    if ts <= tile:
        return [0], 1, tile
    nb = math.ceil((ts - overlap) / (tile - overlap))
    if nb <= 1:
        return [0], 1, tile
    step = (ts - tile) / (nb - 1)
    pos = [int(round(i * step)) for i in range(nb)]
    pos[-1] = ts - tile
    return pos, nb, ts

def pixel_shuffle_down(hr_img, scale):
    h, w, c = hr_img.shape
    new_h, new_w = h // scale, w // scale
    img = hr_img.reshape(new_h, scale, new_w, scale, c)
    img = img.transpose(0, 2, 1, 3, 4)
    img = img.reshape(new_h, new_w, c * scale * scale)
    return np.ascontiguousarray(img)

def build_input(lr_curr, params, lr_prev, packed_hr_prev):
    """Assemble 24-channel input: lr_curr(3) + params(6) + lr_prev(3) + packed_hr_prev(12).
    Then apply INJECT_OVERLAY: for each (ch_idx, value) in INJECT_OVERLAY,
    override that channel of the assembled input with the slider value
    (broadcast spatially). This allows user to inject sliders into channels 3-8.
    """
    channels = [lr_curr, params, lr_prev, packed_hr_prev]
    _dtype = _NP_IO_DTYPE if '_NP_IO_DTYPE' in dir() or '_NP_IO_DTYPE' in globals() else np.float32
    inp = np.ascontiguousarray(np.concatenate(channels, axis=-1), dtype=_dtype)
    # Apply channel overrides from INJECT_OVERLAY (global dict)
    if INJECT_OVERLAY:
        H, W, _ = inp.shape
        for ch_idx, val in INJECT_OVERLAY.items():
            inp[:, :, ch_idx] = val  # broadcast scalar across spatial dims
    inp = inp[None]  # add batch dim
    assert inp.shape[-1] == CHANNELS[SCALE], f"Expected {CHANNELS[SCALE]}ch, got {inp.shape[-1]}"
    return inp

def preprocess_frame(rgb_f32, preblur_val):
    return np.clip(rgb_f32, 0, 1).astype(np.float32)

def get_media_info(path):
    """Get media info (width, height, fps, total_frames).
    For images, uses PIL directly (no ffprobe dependency on image files).
    For videos, uses ffprobe. Handles rotation metadata.
    """
    img_exts = ('.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tiff', '.tif')
    if path.lower().endswith(img_exts):
        try:
            from PIL import Image as _PILImage
            with _PILImage.open(path) as img:
                w, h = img.size
            return w, h, 1.0, 1  # fps=1, total_frames=1 for image
        except Exception as e:
            try:
                img = cv2.imread(path)
                if img is not None:
                    h, w = img.shape[:2]
                    return w, h, 1.0, 1
            except Exception:
                pass
            raise RuntimeError(f"Failed to read image dimensions for {path}: {e}")

    r = subprocess.run(['ffprobe', '-v', 'error', '-select_streams', 'v:0',
                        '-show_entries', 'stream=width,height,r_frame_rate,nb_frames:stream_tags=rotate',
                        '-of', 'json', path], capture_output=True, text=True, timeout=30)
    try:
        data = json.loads(r.stdout)
        if 'streams' not in data or len(data['streams']) == 0:
            raise RuntimeError(f"ffprobe: no video streams found in {path}")
        info = data['streams'][0]
        w, h = int(info['width']), int(info['height'])

        # Check for rotation metadata to get true display dimensions
        rotate = 0
        if 'tags' in info and 'rotate' in info['tags']:
            try:
                rotate = int(float(info['tags']['rotate']))
            except ValueError:
                rotate = 0

        # If rotated 90 or 270 degrees, swap width and height
        if rotate in (90, 270, -90, -270):
            w, h = h, w
            print(f"  ℹ Detected {rotate}° rotation. Display dimensions: {w}x{h}")

        num, den = info['r_frame_rate'].split('/')
        fps = float(num) / float(den) if float(den) > 0 else 25.0
        tf_str = info.get('nb_frames', '0')
        tf = int(tf_str) if tf_str and tf_str.isdigit() and int(tf_str) > 0 else None
        return w, h, fps, tf
    except (json.JSONDecodeError, KeyError, RuntimeError) as e:
        raise RuntimeError(f"ffprobe failed for {path}: {e}\nstderr: {r.stderr}")

def read_image_16bit(path):
    """Read image as 16-bit RGB array (H, W, 3) uint16, preserving original bit depth.
    - 8-bit inputs (JPEG, 8-bit PNG) are upscaled to 16-bit by bit replication
      (val_16 = val_8 * 257 = (val_8 << 8) | val_8) for smooth gradient.
    - 16-bit inputs (16-bit PNG, 16-bit TIFF) are kept as-is.
    - Returns: numpy array (H, W, 3) uint16, RGB order.
    """
    from PIL import Image as _PILImage
    img = _PILImage.open(path)
    mode = img.mode
    if mode in ('I', 'I;16', 'I;16B', 'I;16L'):
        # 16-bit grayscale
        img = img.convert('I;16')
        arr = np.array(img, dtype=np.uint16)
        arr = np.stack([arr, arr, arr], axis=-1)
        input_bit_depth = 16
    elif mode in ('RGB', 'RGBA', 'P', 'L', 'CMYK'):
        if hasattr(img, 'mode') and img.mode == 'RGB':
            try:
                if img.filename and img.filename.lower().endswith(('.png', '.tiff', '.tif')):
                    img_raw = _PILImage.open(path)
                    if img_raw.mode == 'RGB' and 'I;16' in str(img_raw.info.get('mode', '')):
                        img_raw = img_raw.convert('I;16')
                    extrema = img_raw.getextrema()
                    max_val = max(e[1] if isinstance(e, tuple) else e for e in extrema)
                    if max_val > 255:
                        arr_16 = np.array(img_raw, dtype=np.uint16)
                        if arr_16.ndim == 2:
                            arr_16 = np.stack([arr_16, arr_16, arr_16], axis=-1)
                        input_bit_depth = 16
                        return arr_16, input_bit_depth
            except Exception:
                pass
        img_rgb = img.convert('RGB')
        arr_8 = np.array(img_rgb, dtype=np.uint8)
        arr_16 = (arr_8.astype(np.uint16) << 8) | arr_8.astype(np.uint16)
        input_bit_depth = 8
    else:
        img_rgb = img.convert('RGB')
        arr_8 = np.array(img_rgb, dtype=np.uint8)
        arr_16 = (arr_8.astype(np.uint16) << 8) | arr_8.astype(np.uint16)
        input_bit_depth = 8
    return arr_16, input_bit_depth

def compute_resize_dimensions(iw, ih, target_short_side):
    """Compute new (width, height) so SHORT SIDE = target. Preserves orientation.
    Portrait stays portrait, Landscape stays landscape, Square stays square."""
    if target_short_side <= 0:
        return iw, ih
    short_side = min(iw, ih)
    if short_side <= target_short_side:
        return iw, ih
    scale = target_short_side / short_side
    new_w = int(round(iw * scale))
    new_h = int(round(ih * scale))
    new_w -= new_w % 2
    new_h -= new_h % 2
    return new_w, new_h

def detect_orientation(w, h):
    if w > h: return "landscape"
    elif h > w: return "portrait"
    else: return "square"

def resize_frame_16bit(frame_uint16, target_w, target_h):
    h, w = frame_uint16.shape[:2]
    if (w, h) == (target_w, target_h):
        return frame_uint16
    return cv2.resize(frame_uint16, (target_w, target_h), interpolation=cv2.INTER_LANCZOS4)


def apply_recover_original_detail(model_hr, lr_curr, scale, blend=0):
    """Recover Original Detail using cv2 (no PyTorch).
    Extract high-freq detail from original LR, add back to model HR.
    blend: 0-100, 0 = OFF
    """
    if blend == 0:
        return model_hr
    alpha = blend / 100.0
    h, w = lr_curr.shape[:2]
    original_hr = cv2.resize(lr_curr, (w * scale, h * scale), interpolation=cv2.INTER_CUBIC)
    detail = original_hr - cv2.GaussianBlur(original_hr, (5, 5), 1.0)
    return np.clip(model_hr + alpha * detail, 0.0, 1.0)


class FFmpegRGB48Reader:
    def __init__(self, path, orig_w, orig_h, out_w, out_h, fps=25.0, seek_frame=0):
        self.w, self.h = out_w, out_h
        self.frame_bytes = out_w * out_h * 6
        # FIX: Removed '-noautorotate' to allow FFmpeg to physically auto-rotate frames
        cmd = ['ffmpeg', '-v', 'error']
        if seek_frame > 0:
            cmd.extend(['-ss', f'{seek_frame / fps:.6f}'])
        cmd.extend(['-i', path])
        if (orig_w, orig_h) != (out_w, out_h):
            cmd.extend(['-vf', f'scale={out_w}:{out_h}:flags=lanczos'])
        cmd.extend(['-sws_flags', SWS_FLAGS, '-pix_fmt', 'rgb48le', '-vsync', 'passthrough', '-f', 'rawvideo', 'pipe:1'])
        self.proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        self._stderr_lines = []
        def _drain():
            for line in self.proc.stderr:
                self._stderr_lines.append(line)
        threading.Thread(target=_drain, daemon=True).start()
    def read(self):
        data = b''
        while len(data) < self.frame_bytes:
            chunk = self.proc.stdout.read(self.frame_bytes - len(data))
            if not chunk: break
            data += chunk
        if len(data) < self.frame_bytes: return None
        return np.frombuffer(data, dtype=np.uint16).reshape((self.h, self.w, 3)).copy()
    def close(self):
        try: self.proc.stdout.close()
        except: pass
        try:
            self.proc.terminate()
            self.proc.wait(timeout=5)
        except:
            try: self.proc.kill()
            except: pass

if Verbose_Log == "ON":
    print("\n  Functions ready.")

FULL_FEATHER_MASK = make_feather_mask(TILE_H * SCALE, TILE_W * SCALE, OVERLAP * SCALE)

# ═══════════════════════════════════════════════════════════════════════════
# Temporal State
# ═══════════════════════════════════════════════════════════════════════════
prev_lr = None
prev_hr = None

def run_model_tiles(lr_curr, params, lr_prev, packed_hr, tile_h, tile_w, overlap):
    in_h, in_w = lr_curr.shape[:2]
    out_h, out_w = in_h * SCALE, in_w * SCALE

    yp, nh, ph = compute_tile_positions(in_h, tile_h, overlap)
    xp, nw, pw = compute_tile_positions(in_w, tile_w, overlap)

    lr_curr_p = reflect_pad(lr_curr, ph, pw)
    params_p = reflect_pad(params, ph, pw)
    lr_prev_p = reflect_pad(lr_prev, ph, pw)
    packed_hr_p = reflect_pad(packed_hr, ph, pw)

    poh, pow_ = ph * SCALE, pw * SCALE
    obuf = np.zeros((poh, pow_, 3), dtype=np.float32)
    wbuf = np.zeros((poh, pow_), dtype=np.float32)

    for ty in range(nh):
        for tx in range(nw):
            y0, x0 = yp[ty], xp[tx]
            t_curr = lr_curr_p[y0:y0 + tile_h, x0:x0 + tile_w]
            t_params = params_p[y0:y0 + tile_h, x0:x0 + tile_w]
            t_prev = lr_prev_p[y0:y0 + tile_h, x0:x0 + tile_w]
            t_hr = packed_hr_p[y0:y0 + tile_h, x0:x0 + tile_w]

            inp = build_input(t_curr, t_params, t_prev, t_hr)

            # Inference (TRT or ONNX Runtime)
            if _USE_ORT:
                out_tile = ort_sess.run(None, {input_name: inp})[0][0]
            else:
                _out_dtype = _NP_IO_DTYPE if '_NP_IO_DTYPE' in dir() or '_NP_IO_DTYPE' in globals() else np.float32
                cuda.memcpy_htod_async(d_input, inp, stream)
                context.execute_async_v3(stream.handle)
                out_tile = np.empty(output_shape, dtype=_out_dtype)
                cuda.memcpy_dtoh_async(out_tile, d_output, stream)
                stream.synchronize()
                out_tile = out_tile[0] # Remove batch dimension
                # Convert to float32 for downstream processing (clip, feather mask, etc.)
                out_tile = out_tile.astype(np.float32)

            out_tile = np.clip(out_tile, 0, 1)

            oy0, ox0 = y0 * SCALE, x0 * SCALE
            oe, xe = oy0 + tile_h * SCALE, ox0 + tile_w * SCALE
            for c in range(3):
                obuf[oy0:oe, ox0:xe, c] += out_tile[:, :, c] * FULL_FEATHER_MASK
            wbuf[oy0:oe, ox0:xe] += FULL_FEATHER_MASK

    wbuf = np.maximum(wbuf, 1e-6)
    return np.clip(obuf / wbuf[:, :, np.newaxis], 0, 1)[:out_h, :out_w]


def process_frame(input_rgb48, is_warmup=False):
    global prev_lr, prev_hr
    in_h, in_w = input_rgb48.shape[:2]

    # 1. Normalize to 0-1 float32 (RGB format from FFmpeg reader)
    rgb_f32 = input_rgb48.astype(np.float32) / 65535.0

    lr_curr = preprocess_frame(rgb_f32, preBlur_val)

    H, W, _ = lr_curr.shape
    params = np.concatenate([
        np.full((H, W, 1), preBlur_val,     dtype=np.float32),
        np.full((H, W, 1), noise_val,       dtype=np.float32),
        np.full((H, W, 1), details_val,     dtype=np.float32),
        np.full((H, W, 1), halo_val,        dtype=np.float32),
        np.full((H, W, 1), sharpen_val,     dtype=np.float32),
        np.full((H, W, 1), compression_val, dtype=np.float32),
    ], axis=-1)

    if prev_lr is None:
        # First frame (image or video frame #1) — lr_prev = lr_curr (temporal self-reference)
        lr_prev = lr_curr.copy() if USE_TEMPORAL else np.zeros_like(lr_curr)
        hr_simple = cv2.resize(lr_curr, (in_w * SCALE, in_h * SCALE),
                                interpolation=cv2.INTER_CUBIC)
        packed_hr = pixel_shuffle_down(hr_simple, SCALE)
        n_passes = IMAGE_PASSES if IS_IMAGE_INPUT else VIDEO_FIRST_PASSES
        for p in range(n_passes):
            hr_pass = run_model_tiles(lr_curr, params, lr_prev, packed_hr,
                                       TILE_H, TILE_W, OVERLAP)
            hr_pass = np.clip(hr_pass, 0, 1)
            if p < n_passes - 1:
                packed_hr = pixel_shuffle_down(hr_pass, SCALE)
    else:
        # Frame #2+ (video only) — use actual previous frame
        lr_prev = prev_lr if USE_TEMPORAL else np.zeros_like(lr_curr)
        packed_hr = pixel_shuffle_down(prev_hr, SCALE) if USE_TEMPORAL else \
                    pixel_shuffle_down(cv2.resize(lr_curr, (in_w * SCALE, in_h * SCALE),
                                                  interpolation=cv2.INTER_CUBIC), SCALE)

    hr_output = run_model_tiles(lr_curr, params, lr_prev, packed_hr,
                                  TILE_H, TILE_W, OVERLAP)
    hr_output = np.clip(hr_output, 0, 1)

    # Apply Recover Original Detail (cv2-based, NOT PyTorch)
    if CURRENT_RECOVER_ORIGINAL > 0:
        hr_output = apply_recover_original_detail(hr_output, lr_curr, SCALE, CURRENT_RECOVER_ORIGINAL)
        hr_output = np.clip(hr_output, 0, 1)

    if not is_warmup:
        prev_lr = lr_curr.copy()
        prev_hr = hr_output.copy()

    _result = np.ascontiguousarray((hr_output * 65535).astype(np.uint16))
    if _result.max() == 0 and not is_warmup:
        print(f"  ⚠ WARNING: Output is all zeros (black image)!")
        print(f"    Possible causes: TRT engine OOM, input data corrupted, or numpy version mismatch.")
        print(f"    Check: numpy={np.__version__}, VRAM available, or try fewer passes.")
    return _result


# ═══════════════════════════════════════════════════════════════════════════
# Encoder Helpers
# ═══════════════════════════════════════════════════════════════════════════
def setup_encoder(out_w, out_h, fps, out_temp_path, encoder, quality, iha, input_path):
    """Build ffmpeg command and start subprocess for video encoding.
    Returns: (enc_proc, enc_errors, enc_info)
    """
    is_x264_8bit  = (encoder == "x264 8bit")
    is_x265_8bit  = (encoder == "x265 8bit")
    is_x265_10bit = (encoder == "x265 10bit")
    is_prores_12  = (encoder == "ProRes 4444 12bit")
    is_ffv1_16    = (encoder == "FFV1 16bit Lossless")

    video_codec = profile_v = pix_fmt_out = tag_v = input_pix_fmt = vf_filter = None

    if is_x264_8bit:
        # x264 8-bit: input bgr24 (8-bit), output yuv420p 8-bit
        video_codec = "h264_nvenc"
        profile_v = "high"
        pix_fmt_out = "yuv420p"
        tag_v = "avc1"
        input_pix_fmt = "bgr24"
        vf_filter = ""
        enc_cmd = ['ffmpeg', '-y', '-thread_queue_size', '512',
                   '-f', 'rawvideo', '-pix_fmt', input_pix_fmt,
                   '-s', f'{out_w}x{out_h}', '-r', str(fps), '-i', '-']
        if iha:
            enc_cmd.extend(['-thread_queue_size', '512', '-i', input_path])
        enc_cmd.extend(['-map', '0:v:0'])
        if iha:
            enc_cmd.extend(['-map', '1:a:0'])
        enc_cmd.extend(['-sws_flags', SWS_FLAGS, '-color_trc', COLOR_TRC, '-colorspace', COLORSPACE,
                       '-color_primaries', COLOR_PRIMARIES,
                       '-c:v', video_codec, '-profile:v', profile_v, '-pix_fmt', pix_fmt_out,
                       '-preset', 'p7', '-tune', 'hq',
                       '-rc', 'constqp', '-qp', str(quality),
                       '-multipass', '2',
                       '-spatial-aq', '1', '-temporal-aq', '1',
                       '-rc-lookahead', '60', '-tag:v', tag_v])
        if iha:
            enc_cmd.extend(['-c:a', 'copy'])
        enc_cmd.extend(['-movflags', MOVFLAGS, out_temp_path])

    elif is_x265_8bit:
        # x265 8-bit: input bgr24 (8-bit), output yuv420p 8-bit
        video_codec = "hevc_nvenc"
        profile_v = "main"
        pix_fmt_out = "yuv420p"
        tag_v = "hvc1"
        input_pix_fmt = "bgr24"
        vf_filter = ""
        enc_cmd = ['ffmpeg', '-y', '-thread_queue_size', '512',
                   '-f', 'rawvideo', '-pix_fmt', input_pix_fmt,
                   '-s', f'{out_w}x{out_h}', '-r', str(fps), '-i', '-']
        if iha:
            enc_cmd.extend(['-thread_queue_size', '512', '-i', input_path])
        enc_cmd.extend(['-map', '0:v:0'])
        if iha:
            enc_cmd.extend(['-map', '1:a:0'])
        enc_cmd.extend(['-sws_flags', SWS_FLAGS, '-color_trc', COLOR_TRC, '-colorspace', COLORSPACE,
                       '-color_primaries', COLOR_PRIMARIES,
                       '-c:v', video_codec, '-profile:v', profile_v, '-pix_fmt', pix_fmt_out,
                       '-preset', 'p7', '-tune', 'hq',
                       '-rc', 'constqp', '-qp', str(quality),
                       '-multipass', '2',
                       '-spatial-aq', '1', '-temporal-aq', '1',
                       '-rc-lookahead', '60', '-tag:v', tag_v])
        if iha:
            enc_cmd.extend(['-c:a', 'copy'])
        enc_cmd.extend(['-movflags', MOVFLAGS, out_temp_path])

    elif is_x265_10bit:
        # x265 10-bit: input bgr48le (16-bit BGR), output p010le 10-bit
        video_codec = "hevc_nvenc"
        profile_v = "main10"
        pix_fmt_out = "p010le"
        tag_v = "hvc1"
        input_pix_fmt = "bgr48le"
        vf_filter = "format=p010le"
        enc_cmd = ['ffmpeg', '-y', '-thread_queue_size', '512',
                   '-f', 'rawvideo', '-pix_fmt', input_pix_fmt,
                   '-s', f'{out_w}x{out_h}', '-r', str(fps), '-i', '-']
        if iha:
            enc_cmd.extend(['-thread_queue_size', '512', '-i', input_path])
        enc_cmd.extend(['-map', '0:v:0'])
        if iha:
            enc_cmd.extend(['-map', '1:a:0'])
        enc_cmd.extend(['-vf', vf_filter,
                       '-sws_flags', SWS_FLAGS, '-color_trc', COLOR_TRC, '-colorspace', COLORSPACE,
                       '-color_primaries', COLOR_PRIMARIES,
                       '-c:v', video_codec, '-profile:v', profile_v, '-pix_fmt', pix_fmt_out,
                       '-preset', 'p7', '-tune', 'hq',
                       '-rc', 'constqp', '-qp', str(quality),
                       '-multipass', '2',
                       '-spatial-aq', '1', '-temporal-aq', '1',
                       '-rc-lookahead', '60', '-tag:v', tag_v])
        if iha:
            enc_cmd.extend(['-c:a', 'copy'])
        enc_cmd.extend(['-movflags', MOVFLAGS, out_temp_path])

    elif is_prores_12:
        # ProRes 4444 12-bit: Convert Full Range RGB to Limited Range YUV 4:4:4:4 12-bit
        # FIX: Using yuv444p12le instead of gbrp12le and explicit color range to fix dark/contrast output
        bits_per_mb = int(800 - (quality / 51.0) * 700)  # Q=0 → 800, Q=51 → 100
        bits_per_mb = max(100, min(800, bits_per_mb))
        video_codec = "prores_ks"
        profile_v = "4"  # 4 = ProRes 4444
        pix_fmt_out = "yuv444p12le"
        input_pix_fmt = "rgb48le"
        # Explicitly define color range conversion: Full RGB -> Limited TV Range YUV
        vf_filter = "scale=in_range=full:in_color_matrix=bt709:out_range=tv:out_color_matrix=bt709"
        enc_cmd = ['ffmpeg', '-y', '-thread_queue_size', '512',
                   '-f', 'rawvideo', '-pix_fmt', input_pix_fmt,
                   '-s', f'{out_w}x{out_h}', '-r', str(fps), '-i', '-']
        if iha:
            enc_cmd.extend(['-thread_queue_size', '512', '-i', input_path])
        enc_cmd.extend(['-map', '0:v:0'])
        if iha:
            enc_cmd.extend(['-map', '1:a:0'])
        enc_cmd.extend(['-vf', vf_filter,
                       '-sws_flags', SWS_FLAGS, '-color_trc', COLOR_TRC, '-colorspace', COLORSPACE,
                       '-color_primaries', COLOR_PRIMARIES, '-color_range', 'tv',
                       '-c:v', video_codec, '-profile:v', profile_v, '-pix_fmt', pix_fmt_out,
                       '-bits_per_mb', str(bits_per_mb)])
        if iha:
            enc_cmd.extend(['-c:a', 'pcm_s16le'])
        enc_cmd.extend(['-movflags', MOVFLAGS, out_temp_path])

    elif is_ffv1_16:
        # FFV1 16-bit lossless: input rgb48le, output rgb48le (true lossless)
        video_codec = "ffv1"
        profile_v = ""
        pix_fmt_out = "rgb48le"
        tag_v = ""
        input_pix_fmt = "rgb48le"
        vf_filter = ""
        enc_cmd = ['ffmpeg', '-y', '-thread_queue_size', '512',
                   '-f', 'rawvideo', '-pix_fmt', 'rgb48le',
                   '-s', f'{out_w}x{out_h}', '-r', str(fps), '-i', '-']
        if iha:
            enc_cmd.extend(['-thread_queue_size', '512', '-i', input_path])
        enc_cmd.extend(['-map', '0:v:0'])
        if iha:
            enc_cmd.extend(['-map', '1:a:0'])
        enc_cmd.extend(['-c:v', 'ffv1', '-level', '3', '-coder', '1', '-context', '1',
                       '-g', '300', '-slices', '16', '-slicecrc', '1', '-pix_fmt', 'rgb48le'])
        if iha:
            enc_cmd.extend(['-c:a', 'flac'])
        enc_cmd.append(out_temp_path)

    else:
        raise RuntimeError(f"Unknown encoder: {encoder}")

    enc_proc = subprocess.Popen(enc_cmd, stdin=subprocess.PIPE, stderr=subprocess.PIPE, bufsize=0)
    enc_errors = []
    def _read_enc_stderr():
        for line in enc_proc.stderr:
            enc_errors.append(line.decode('utf-8', errors='ignore'))
    threading.Thread(target=_read_enc_stderr, daemon=True).start()

    time.sleep(2)
    if enc_proc.poll() is not None:
        err = ''.join(enc_errors[-20:])
        raise RuntimeError(f"Encoder failed to start!\nFFmpeg stderr:\n{err}")

    enc_info = {
        'is_x264_8bit': is_x264_8bit,
        'is_x265_8bit': is_x265_8bit,
        'is_x265_10bit': is_x265_10bit,
        'is_prores_12': is_prores_12,
        'is_ffv1_16': is_ffv1_16,
        'video_codec': video_codec,
        'profile_v': profile_v,
        'pix_fmt_out': pix_fmt_out,
        'tag_v': tag_v,
        'input_pix_fmt': input_pix_fmt,
        'vf_filter': vf_filter,
    }
    return enc_proc, enc_errors, enc_info


def write_frame_to_encoder(enc_proc, enc_info, result_uint16_rgb):
    """Write a processed uint16 RGB frame to the encoder subprocess.
    FIX: ProRes uses rgb48le input → write RGB (NOT BGR), like FFV1.
    """
    if enc_info['is_x264_8bit'] or enc_info['is_x265_8bit']:
        # 8-bit encoders: convert uint16 → uint8, write 8-bit BGR
        result_8bit = cv2.cvtColor(
            (result_uint16_rgb >> 8).astype(np.uint8),
            cv2.COLOR_RGB2BGR
        )
        enc_proc.stdin.write(result_8bit.tobytes())
    elif enc_info['is_x265_10bit']:
        # x265 10-bit: input bgr48le (16-bit BGR), output p010le
        result_uint16_bgr = cv2.cvtColor(result_uint16_rgb, cv2.COLOR_RGB2BGR)
        enc_proc.stdin.write(np.ascontiguousarray(result_uint16_bgr, dtype=np.uint16).tobytes())
    elif enc_info['is_prores_12']:
        # ProRes 4444 12-bit: input rgb48le (16-bit RGB) — FIX: write RGB, NOT BGR
        enc_proc.stdin.write(np.ascontiguousarray(result_uint16_rgb, dtype=np.uint16).tobytes())
    elif enc_info['is_ffv1_16']:
        # FFV1 16-bit lossless: input rgb48le, write 16-bit RGB
        enc_proc.stdin.write(np.ascontiguousarray(result_uint16_rgb, dtype=np.uint16).tobytes())
    enc_proc.stdin.flush()


def has_audio(path):
    try:
        r = subprocess.run(['ffprobe', '-v', 'error', '-select_streams', 'a',
                            '-show_entries', 'stream=codec_type',
                            '-of', 'csv=p=0', path],
                           capture_output=True, text=True, timeout=10)
        return 'audio' in r.stdout
    except:
        return False


def process_video_pass(input_path, output_path, encoder, quality,
                       apply_input_resize=True, preflight=True, pass_name="Pass",
                       audio_source_path=None, output_resize=True, is_final_pass=True):
    """Run a full video upscale pass.

    input_path: source video file
    output_path: output video file (with proper extension)
    encoder: encoder name (e.g., "FFV1 16bit Lossless", "x265 10bit")
    quality: QP/CRF value
    apply_input_resize: if True, apply Input_Video_Resolution resize to input
    preflight: if True, run preflight warmup
    pass_name: label for printout
    audio_source_path: path for audio source (default: input_path)
    output_resize: if True, apply Output_Video_Resize to AI output
    is_final_pass: if True, honor Save_Frames mode; otherwise always direct encode
    """
    global prev_lr, prev_hr

    if audio_source_path is None:
        audio_source_path = input_path

    iw, ih, fps, tf = get_media_info(input_path)

    print(f"\n{'='*70}")
    print(f"  {pass_name}")
    print(f"{'='*70}")
    print(f"  Input: {input_path}")
    print(f"  Resolution: {iw}x{ih} ({detect_orientation(iw, ih)})")
    print(f"  FPS: {fps:.2f}")
    if tf:
        print(f"  Total Frames: {tf}")
    try:
        in_size = os.path.getsize(input_path) / (1024*1024)
        print(f"  File Size: {in_size:.1f} MB")
    except:
        pass

    # === Input Resize (orientation-preserving, always log) ===
    if apply_input_resize:
        pw, ph = compute_resize_dimensions(iw, ih, MAX_INPUT_HEIGHT)
    else:
        pw, ph = iw, ih
    print(f"  Input Resize: {iw}x{ih} → {pw}x{ph} ({detect_orientation(pw, ph)})")

    # === Output Resize (post-AI-upscale) — VIDEO only ===
    ai_out_w, ai_out_h = pw * SCALE, ph * SCALE
    if output_resize and OUTPUT_TARGET_SHORT_SIDE > 0:
        out_w, out_h = compute_resize_dimensions(ai_out_w, ai_out_h, OUTPUT_TARGET_SHORT_SIDE)
    else:
        out_w, out_h = ai_out_w, ai_out_h
    if (out_w, out_h) != (ai_out_w, ai_out_h):
        print(f"  Output Resize: {ai_out_w}x{ai_out_h} → {out_w}x{out_h} ({detect_orientation(out_w, out_h)})")
    else:
        print(f"  Output Resize: {ai_out_w}x{ai_out_h} (no resize)")

    print(f"  AI Pipeline: {pw}x{ph} → {ai_out_w}x{ai_out_h} (AI {SCALE}x) → {out_w}x{out_h} (final)")

    # === Audio detection (from audio_source_path, not input_path) ===
    iha = has_audio(audio_source_path)

    # === Save_Frames mode (only honored for final pass) ===
    FD = '/content/upscaled_frames'
    use_save_frames = (Save_Frames == "ON") and is_final_pass
    if use_save_frames:
        shutil.rmtree(FD, ignore_errors=True)
        os.makedirs(FD)
        print("  ℹ Save_Frames='ON': saving 16-bit PNGs (quality preserved)")

    # === Setup encoder (only if NOT using save_frames) ===
    enc_proc = None
    enc_errors = []
    enc_info = None
    if not use_save_frames:
        if Verbose_Log == "ON":
            print(f"\n  Direct encoding: {encoder}")
        else:
            print(f"  Direct encoding: {encoder}")
        enc_proc, enc_errors, enc_info = setup_encoder(
            out_w, out_h, fps, output_path, encoder, quality, iha, audio_source_path
        )
        if Verbose_Log == "ON":
            print(f"  Encoder started (rc-lookahead=60)")

    # === Preflight warmup ===
    if preflight and PREFLIGHT > 0:
        if Verbose_Log == "ON":
            print(f"\n  Preflight warmup ({PREFLIGHT} frames)...")
        else:
            print("\nSystem Engine Starting...")
        pre_reader = FFmpegRGB48Reader(input_path, iw, ih, pw, ph, fps, 0)
        wf = []
        for _ in range(PREFLIGHT):
            f = pre_reader.read()
            if f is None: break
            wf.append(f)
        pre_reader.close()
        for w in wf[:PREFLIGHT]:
            process_frame(w, is_warmup=True)
        prev_lr = None
        prev_hr = None
        if Verbose_Log == "ON":
            print("  Preflight complete.\n")
        else:
            print("System Engine Started\n")

    # === Main loop ===
    reader = FFmpegRGB48Reader(input_path, iw, ih, pw, ph, fps, seek_frame=0)
    fi = 0
    st0 = time.time()
    stopped = False

    try:
        while True:
            frame = reader.read()
            if frame is None: break
            fi += 1
            t0 = time.time()

            # process_frame returns uint16 [0,65535] RGB
            result_uint16_rgb = process_frame(frame)
            result_uint16_rgb = np.ascontiguousarray(result_uint16_rgb, dtype=np.uint16)

            # === Output Resize (post-AI-upscale, pre-encoder) ===
            if (out_w, out_h) != (ai_out_w, ai_out_h):
                result_uint16_rgb = resize_frame_16bit(result_uint16_rgb, out_w, out_h)
                result_uint16_rgb = np.ascontiguousarray(result_uint16_rgb, dtype=np.uint16)

            if use_save_frames:
                result_uint16_bgr = cv2.cvtColor(result_uint16_rgb, cv2.COLOR_RGB2BGR)
                result_uint16_bgr = np.ascontiguousarray(result_uint16_bgr, dtype=np.uint16)
                cv2.imwrite(os.path.join(FD, f'f{fi:06d}.png'), result_uint16_bgr)
            else:
                try:
                    write_frame_to_encoder(enc_proc, enc_info, result_uint16_rgb)
                except BrokenPipeError:
                    err = ''.join(enc_errors[-30:])
                    raise RuntimeError(f"Pipe broken frame {fi}!\nstderr:\n{err}")

            el = time.time() - st0
            fps2 = fi / el if el > 0 else 0
            if tf is None or fi > tf: tf = fi
            if tf and fps2 > 0:
                eta = (tf - fi) / fps2
                print(f"\r  {pass_name} Frame {fi:>5}/{tf} ({fi/tf*100:.1f}%) | "
                      f"{time.time()-t0:.2f}s/f | {fps2:.2f}fps | "
                      f"ETA:{eta/60:.1f}m", end='', flush=True)
            else:
                print(f"\r  {pass_name} Frame {fi} | {time.time()-t0:.2f}s/f | {fps2:.2f}fps",
                      end='', flush=True)
    except KeyboardInterrupt:
        print(f"\n\n  ⏹ Stopped by user at frame {fi}!")
        print(f"  Saving {fi} frames processed so far...")
        stopped = True
    except Exception as e:
        print(f"\n  ERROR: {e}")
        raise
    finally:
        try:
            reader.close()
        except:
            pass
        if enc_proc:
            try: enc_proc.stdin.flush()
            except: pass
            try: enc_proc.stdin.close()
            except: pass
            try:
                ret = enc_proc.wait(timeout=120)
                if ret != 0:
                    err = ''.join(enc_errors[-30:])
                    print(f"\n  Encoder warning (exit {ret}):")
                    if err:
                        print(err[-500:])
            except subprocess.TimeoutExpired:
                print(f"\n  ⚠ Encoder timed out, killing...")
                enc_proc.kill()
                enc_proc.wait()
        print()

    return fi, fps, tf, stopped, out_w, out_h


def lossless_resize_video(input_path, output_path, target_resolution):
    """Lossless resize of a video file to target short side (FFV1 rgb48le).
    Used between Pass 1 and Pass 2 of 2-pass video upscale.
    """
    iw, ih, fps, tf = get_media_info(input_path)
    target_short = _INPUT_RES_MAP.get(target_resolution, 99999)
    new_w, new_h = compute_resize_dimensions(iw, ih, target_short)

    print(f"\n  📐 Lossless Resize (Pass1 → Pass2 input):")
    print(f"     Input:  {iw}x{ih}")
    print(f"     Target: {target_resolution} (short side ≤ {target_short}p)")
    print(f"     → Resized to: {new_w}x{new_h} ({detect_orientation(new_w, new_h)}) [orientation preserved]")

    cmd = ['ffmpeg', '-y', '-i', input_path,
           '-vf', f'scale={new_w}:{new_h}:flags=lanczos',
           '-sws_flags', SWS_FLAGS,
           '-c:v', 'ffv1', '-level', '3', '-coder', '1', '-context', '1',
           '-g', '300', '-slices', '16', '-slicecrc', '1', '-pix_fmt', 'rgb48le',
           '-an',
           output_path]
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=7200)
    if r.returncode != 0:
        print(f"  FFmpeg error: {r.stderr[-500:]}")
        raise RuntimeError("Lossless resize failed")


def get_output_extension(encoder):
    """Get file extension for encoder."""
    if encoder == "FFV1 16bit Lossless":
        return '.mkv'
    elif encoder == "ProRes 4444 12bit":
        return '.mov'
    else:
        return '.mp4'


def remux_audio(temp_path, final_path, audio_source_path):
    """Remux video with audio from audio_source_path → final_path."""
    iha = has_audio(audio_source_path)
    # Build command: all -i inputs first, then output options
    cmd = ['ffmpeg', '-y']
    cmd.extend(['-i', temp_path])  # input 0: video
    if iha:
        cmd.extend(['-i', audio_source_path])  # input 1: audio
    cmd.extend(['-c:v', 'copy', '-map', '0:v'])
    if iha:
        cmd.extend(['-c:a', 'copy', '-map', '1:a:0'])
    cmd.extend(['-map_metadata', '0', '-movflags', 'use_metadata_tags', final_path])
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
    if r.returncode != 0:
        print(f"  Audio remux warning: {r.stderr[-300:]}")


def encode_from_saved_frames(FD, fi, fps, out_temp_path, encoder, quality, input_path):
    """Encode final video from saved 16-bit PNG frames (Save_Frames mode)."""
    print(f"\n  Encoding video from saved frames ({fi} frames, {fps:.2f} fps)...")
    vd = fi / fps if fps > 0 else 0
    iha = has_audio(input_path)

    if encoder == "FFV1 16bit Lossless":
        cmd = ['ffmpeg', '-y', '-framerate', str(fps), '-i', os.path.join(FD, 'f%06d.png'),
               '-i', input_path, '-map', '0:v:0']
        if iha:
            cmd.extend(['-map', '1:a:0', '-c:a', 'flac'])
        cmd.extend(['-c:v', 'ffv1', '-level', '3', '-coder', '1', '-context', '1',
                   '-g', '300', '-slices', '16', '-slicecrc', '1', '-pix_fmt', 'rgb48le'])
        cmd.extend(['-movflags', MOVFLAGS, '-t', str(vd), '-map_metadata', '0', out_temp_path])

    elif encoder == "ProRes 4444 12bit":
        cmd = ['ffmpeg', '-y', '-framerate', str(fps), '-i', os.path.join(FD, 'f%06d.png'),
               '-i', input_path, '-map', '0:v:0']
        if iha:
            cmd.extend(['-map', '1:a:0', '-c:a', 'pcm_s16le'])
        cmd.extend(['-vf', 'scale=in_range=full:in_color_matrix=bt709:out_range=tv:out_color_matrix=bt709',
                    '-sws_flags', SWS_FLAGS, '-color_trc', COLOR_TRC, '-colorspace', COLORSPACE, '-color_primaries', COLOR_PRIMARIES,
                    '-color_range', 'tv',
                    '-c:v', 'prores_ks', '-profile:v', '4', '-pix_fmt', 'yuv444p12le', '-bits_per_mb', '800'])
        cmd.extend(['-movflags', MOVFLAGS, '-t', str(vd), '-map_metadata', '0', out_temp_path])

    elif encoder == "x265 10bit":
        cmd = ['ffmpeg', '-y', '-framerate', str(fps), '-i', os.path.join(FD, 'f%06d.png'),
               '-i', input_path, '-map', '0:v:0', '-sws_flags', SWS_FLAGS,
               '-color_trc', COLOR_TRC, '-colorspace', COLORSPACE, '-color_primaries', COLOR_PRIMARIES,
               '-vf', 'format=p010le',
               '-c:v', 'hevc_nvenc', '-profile:v', 'main10', '-pix_fmt', 'p010le',
               '-preset', 'p7', '-tune', 'hq',
               '-rc', 'constqp', '-qp', str(quality),
               '-multipass', '2',
               '-spatial-aq', '1', '-temporal-aq', '1',
               '-rc-lookahead', '60', '-tag:v', 'hvc1']
        if iha:
            cmd.extend(['-map', '1:a:0', '-c:a', 'copy'])
        cmd.extend(['-movflags', MOVFLAGS, '-t', str(vd), '-map_metadata', '0', out_temp_path])

    elif encoder == "x265 8bit":
        cmd = ['ffmpeg', '-y', '-framerate', str(fps), '-i', os.path.join(FD, 'f%06d.png'),
               '-i', input_path, '-map', '0:v:0', '-sws_flags', SWS_FLAGS,
               '-color_trc', COLOR_TRC, '-colorspace', COLORSPACE, '-color_primaries', COLOR_PRIMARIES,
               '-c:v', 'hevc_nvenc', '-profile:v', 'main', '-pix_fmt', 'yuv420p',
               '-preset', 'p7', '-tune', 'hq',
               '-rc', 'constqp', '-qp', str(quality),
               '-multipass', '2',
               '-spatial-aq', '1', '-temporal-aq', '1',
               '-rc-lookahead', '60', '-tag:v', 'hvc1']
        if iha:
            cmd.extend(['-map', '1:a:0', '-c:a', 'copy'])
        cmd.extend(['-movflags', MOVFLAGS, '-t', str(vd), '-map_metadata', '0', out_temp_path])

    elif encoder == "x264 8bit":
        cmd = ['ffmpeg', '-y', '-framerate', str(fps), '-i', os.path.join(FD, 'f%06d.png'),
               '-i', input_path, '-map', '0:v:0', '-sws_flags', SWS_FLAGS,
               '-color_trc', COLOR_TRC, '-colorspace', COLORSPACE, '-color_primaries', COLOR_PRIMARIES,
               '-c:v', 'h264_nvenc', '-profile:v', 'high', '-pix_fmt', 'yuv420p',
               '-preset', 'p7', '-tune', 'hq',
               '-rc', 'constqp', '-qp', str(quality),
               '-multipass', '2',
               '-spatial-aq', '1', '-temporal-aq', '1',
               '-rc-lookahead', '60', '-tag:v', 'avc1']
        if iha:
            cmd.extend(['-map', '1:a:0', '-c:a', 'copy'])
        cmd.extend(['-movflags', MOVFLAGS, '-t', str(vd), '-map_metadata', '0', out_temp_path])
    else:
        raise RuntimeError(f"Unknown encoder for Save_Frames mode: {encoder}")

    r = subprocess.run(cmd, capture_output=True, text=True, timeout=7200)
    if r.returncode != 0:
        print(f"  FFmpeg error: {r.stderr[-500:]}")
        raise RuntimeError("FFmpeg encode from saved frames failed")


# ═══════════════════════════════════════════════════════════════════════════
# Main Processing
# ═══════════════════════════════════════════════════════════════════════════
if not os.path.exists(INPUT_PATH):
    raise FileNotFoundError(INPUT_PATH)

bn = os.path.splitext(os.path.basename(INPUT_PATH))[0]
odir = os.path.dirname(INPUT_PATH) or '/content'

if IS_IMAGE_INPUT:
    # ═══════════════════════════════════════════════════════════════════════════
    # IMAGE PROCESSING (supports 1-pass and 2-pass presets)
    # ═══════════════════════════════════════════════════════════════════════════
    iw, ih, _, _ = get_media_info(INPUT_PATH)
    print(f"\n  📷 Input Image: {iw}x{ih} ({detect_orientation(iw, ih)})")
    rw, rh = compute_resize_dimensions(iw, ih, MAX_IMAGE_INPUT_DIM)
    needs_downscale = (rw, rh) != (iw, ih)
    print(f"  Input Resize: {iw}x{ih} → {rw}x{rh} ({detect_orientation(rw, rh)})")

    print(f"\n  Reading image (16-bit safe)...")
    frame_16bit_rgb, input_bit_depth = read_image_16bit(INPUT_PATH)
    print(f"  Input bit depth: {input_bit_depth}-bit, shape: {frame_16bit_rgb.shape}")
    frame = frame_16bit_rgb

    if frame is None:
        raise RuntimeError("Failed to read image")

    if needs_downscale:
        frame = resize_frame_16bit(frame, rw, rh)
        print(f"  → Downscaled to {frame.shape[1]}x{frame.shape[0]} ({detect_orientation(rw, rh)})")

    if PASS_COUNT == 1:
        # === SINGLE PASS (Custom or V1 Anime) ===
        print(f"\n  Processing image ({frame.shape[1]}x{frame.shape[0]}) with {IMAGE_PASSES} passes...")
        result_uint16_rgb = process_frame(frame)
    else:
        # === 2-PASS IMAGE (V2/V3 presets) ===
        # Pass 1: Upscale with Pass1 sliders → save intermediate as uint16
        print(f"\n{'#'*60}")
        print(f"# PASS 1 of 2 (Image)")
        print(f"# Sliders: AAD={PASS1_SLIDERS['anti_alias_deblur']}, RN={PASS1_SLIDERS['reduce_noise']}, RD={PASS1_SLIDERS['recover_details']}")
        print(f"{'#'*60}")
        set_pass_params(PASS1_SLIDERS)
        prev_lr = None
        prev_hr = None
        print(f"  Processing Pass 1 ({frame.shape[1]}x{frame.shape[0]}) with {IMAGE_PASSES} passes...")
        pass1_result = process_frame(frame)
        print(f"  Pass 1 done: shape={pass1_result.shape}")

        # Resize Pass 1 result back to input size for Pass 2
        # (2-pass image: Pass1 upscale → resize to input → Pass2 upscale)
        pass1_h, pass1_w = pass1_result.shape[:2]
        target_h, target_w = frame.shape[:2]
        print(f"  Resizing Pass 1 output: {pass1_w}x{pass1_h} → {target_w}x{target_h}")
        pass1_resized = resize_frame_16bit(pass1_result, target_w, target_h)
        pass1_resized = np.ascontiguousarray(pass1_resized, dtype=np.uint16)

        # Pass 2: Upscale resized Pass 1 result with Pass2 sliders
        print(f"\n{'#'*60}")
        print(f"# PASS 2 of 2 (Image)")
        print(f"# Sliders: AAD={PASS2_SLIDERS['anti_alias_deblur']}, RN={PASS2_SLIDERS['reduce_noise']}, RD={PASS2_SLIDERS['recover_details']}")
        print(f"{'#'*60}")
        set_pass_params(PASS2_SLIDERS)
        prev_lr = None
        prev_hr = None
        print(f"  Processing Pass 2 ({pass1_resized.shape[1]}x{pass1_resized.shape[0]}) with {IMAGE_PASSES} passes...")
        result_uint16_rgb = process_frame(pass1_resized)
        print(f"  Pass 2 done: shape={result_uint16_rgb.shape}")

    # === SAFE uint16 save (numpy 1.x and 2.x compatible) ===
    result_uint16_rgb = np.ascontiguousarray(result_uint16_rgb, dtype=np.uint16)
    img_bgr = cv2.cvtColor(result_uint16_rgb, cv2.COLOR_RGB2BGR)
    img_bgr = np.ascontiguousarray(img_bgr, dtype=np.uint16)

    print(f"  uint16 array: dtype={img_bgr.dtype}, shape={img_bgr.shape}, min={img_bgr.min()}, max={img_bgr.max()}, mean={img_bgr.mean():.1f}")

    OUT_BASE = os.path.join(odir, f"KZM.FX.{bn}_proteus_v3")
    if Image_Format == "PNG 8bit":
        out_path = OUT_BASE + '_8bit.png'
        img_8 = (img_bgr >> 8).astype(np.uint8)
        img_8 = np.ascontiguousarray(img_8)
        cv2.imwrite(out_path, img_8)
    elif Image_Format == "PNG 16bit":
        out_path = OUT_BASE + '_16bit.png'
        ok = cv2.imwrite(out_path, img_bgr)
        if not ok or os.path.getsize(out_path) < 1000:
            print(f"  cv2.imwrite failed, using PIL fallback...")
            from PIL import Image as _PILImg
            cv2.imwrite(out_path, img_bgr, [cv2.IMWRITE_PNG_COMPRESSION, 0])
            if not os.path.exists(out_path) or os.path.getsize(out_path) < 1000:
                np.save(out_path.replace('.png', '.npy'), img_bgr.astype(np.float32))
                print(f"  WARNING: Saved as .npy fallback (16-bit PNG write failed)")
                out_path = out_path.replace('.png', '.npy')
    elif Image_Format == "TIFF 10bit":
        out_path = OUT_BASE + '_10bit.tiff'
        img_10 = np.right_shift(img_bgr, 6).astype(np.uint16)
        img_10 = np.left_shift(img_10, 6).astype(np.uint16)
        img_10 = np.ascontiguousarray(img_10)
        ok = cv2.imwrite(out_path, img_10)
        if not ok or os.path.getsize(out_path) < 1000:
            from PIL import Image as _PILImg
            _pil_img = _PILImg.fromarray(img_10[:, :, 0], mode='I;16')
            _pil_img.save(out_path, format='TIFF')
    elif Image_Format == "TIFF 12bit":
        out_path = OUT_BASE + '_12bit.tiff'
        img_12 = np.right_shift(img_bgr, 4).astype(np.uint16)
        img_12 = np.left_shift(img_12, 4).astype(np.uint16)
        img_12 = np.ascontiguousarray(img_12)
        ok = cv2.imwrite(out_path, img_12)
        if not ok or os.path.getsize(out_path) < 1000:
            from PIL import Image as _PILImg
            _pil_img = _PILImg.fromarray(img_12[:, :, 0], mode='I;16')
            _pil_img.save(out_path, format='TIFF')
    elif Image_Format == "TIFF 16bit":
        out_path = OUT_BASE + '_16bit.tiff'
        ok = cv2.imwrite(out_path, img_bgr)
        if not ok or os.path.getsize(out_path) < 1000:
            from PIL import Image as _PILImg
            _pil_img = _PILImg.fromarray(img_bgr[:, :, 0], mode='I;16')
            _pil_img.save(out_path, format='TIFF')
    else:
        out_path = OUT_BASE + '_16bit.png'
        cv2.imwrite(out_path, img_bgr)

    # Verify output file
    if os.path.exists(out_path):
        _fsize = os.path.getsize(out_path)
        if _fsize < 5000:
            print(f"  WARNING: Output file is very small ({_fsize} bytes) — may be corrupt!")
            print(f"  This usually means numpy 2.x is installed. Check Cell 1 output.")
        print(f"  Image saved: {out_path}")
        print(f"  Format: {Image_Format}, Size: {_fsize/1024:.1f} KB")
        print(f"  Output Resolution: {img_bgr.shape[1]}x{img_bgr.shape[0]} ({detect_orientation(img_bgr.shape[1], img_bgr.shape[0])})")
    else:
        print(f"  ERROR: Output file was not created!")
        out_path = None

    if out_path and os.path.exists(out_path):
        if Auto_Download == "ON":
            try:
                from google.colab import files
                files.download(out_path)
                print(f"  ✓ Auto download started: {os.path.basename(out_path)}")
            except Exception as _ad_err:
                print(f"  ⚠ Auto download failed: {_ad_err}")
                print(f"  File at: {out_path}")
        else:
            print(f"  (Auto download disabled - file at {out_path})")

else:
    # ═══════════════════════════════════════════════════════════════════════════
    # VIDEO PROCESSING
    # ═══════════════════════════════════════════════════════════════════════════
    OUT_EXT = get_output_extension(Encoder)
    OUT = os.path.join(odir, f"KZM.FX.{bn}_proteus_v3{OUT_EXT}")
    OUT_TEMP = os.path.join(odir, f"KZM.FX.{bn}_proteus_v3_temp{OUT_EXT}")

    if PASS_COUNT == 1:
        # === Single-pass (V1 Anime 1 Pass or Custom) ===
        fi, fps, tf, stopped, out_w, out_h = process_video_pass(
            INPUT_PATH, OUT_TEMP, Encoder, Quality,
            apply_input_resize=True, preflight=True, pass_name="Processing",
            audio_source_path=INPUT_PATH, output_resize=True, is_final_pass=True
        )

        # If Save_Frames mode was used, encode video from saved PNGs
        if Save_Frames == "ON":
            FD = '/content/upscaled_frames'
            encode_from_saved_frames(FD, fi, fps, OUT_TEMP, Encoder, Quality, INPUT_PATH)

        # Remux audio from original input
        remux_audio(OUT_TEMP, OUT, INPUT_PATH)
        # If remux failed (no audio or error), just copy temp → final
        if not os.path.exists(OUT) and os.path.exists(OUT_TEMP):
            shutil.move(OUT_TEMP, OUT)
            print(f"  (Moved temp → final: remux failed, using temp)")
        try: os.remove(OUT_TEMP)
        except: pass

    else:
        # === 2-pass (V2 Anime 2 Pass or V3 Anime High Quality) ===
        # PASS 1: FFV1 16bit Lossless intermediate (no output resize, no Save_Frames)
        temp_pass1 = os.path.join(odir, f"KZM.FX.{bn}_proteus_v3_pass1.mkv")
        set_pass_params(PASS1_SLIDERS)
        print(f"\n{'#'*70}")
        print(f"# PASS 1 of 2 — FFV1 16bit Lossless intermediate")
        print(f"# Sliders: AAD={PASS1_SLIDERS['anti_alias_deblur']}, RN={PASS1_SLIDERS['reduce_noise']}, "
              f"RD={PASS1_SLIDERS['recover_details']}, DH={PASS1_SLIDERS['dehalo']}, "
              f"SH={PASS1_SLIDERS['sharpen']}, RC={PASS1_SLIDERS['revert_compression']}")
        print(f"{'#'*70}")
        fi1, fps1, tf1, stopped1, _, _ = process_video_pass(
            INPUT_PATH, temp_pass1, "FFV1 16bit Lossless", 0,
            apply_input_resize=True, preflight=True, pass_name="Pass 1",
            audio_source_path=INPUT_PATH, output_resize=False, is_final_pass=False
        )

        if stopped1:
            print(f"\n  ⏹ Pass 1 stopped at frame {fi1}. Aborting 2-pass.")
            # Save Pass 1 output as final (with audio)
            remux_audio(temp_pass1, OUT, INPUT_PATH)
            try: os.remove(temp_pass1)
            except: pass
            fi, fps, tf, stopped = fi1, fps1, tf1, True
            iw_p1, ih_p1, _, _ = get_media_info(INPUT_PATH)
            pw_p1, ph_p1 = compute_resize_dimensions(iw_p1, ih_p1, MAX_INPUT_HEIGHT)
            out_w, out_h = pw_p1 * SCALE, ph_p1 * SCALE
        else:
            # LOSSLESS RESIZE: Pass1 → Pass2 input (using Input_Video_Resolution)
            temp_resized = os.path.join(odir, f"KZM.FX.{bn}_proteus_v3_pass1_resized.mkv")
            lossless_resize_video(temp_pass1, temp_resized, Input_Video_Resolution)
            try: os.remove(temp_pass1)
            except: pass

            # PASS 2: Final encoder with Pass2 sliders
            set_pass_params(PASS2_SLIDERS)
            print(f"\n{'#'*70}")
            print(f"# PASS 2 of 2 — Final encoder: {Encoder}")
            print(f"# Sliders: AAD={PASS2_SLIDERS['anti_alias_deblur']}, RN={PASS2_SLIDERS['reduce_noise']}, "
                  f"RD={PASS2_SLIDERS['recover_details']}, DH={PASS2_SLIDERS['dehalo']}, "
                  f"SH={PASS2_SLIDERS['sharpen']}, RC={PASS2_SLIDERS['revert_compression']}")
            print(f"{'#'*70}")
            fi2, fps2, tf2, stopped2, out_w, out_h = process_video_pass(
                temp_resized, OUT_TEMP, Encoder, Quality,
                apply_input_resize=False, preflight=True, pass_name="Pass 2",
                audio_source_path=INPUT_PATH, output_resize=True, is_final_pass=True
            )

            # If Save_Frames mode was used in Pass 2, encode video from saved PNGs
            if Save_Frames == "ON":
                FD = '/content/upscaled_frames'
                encode_from_saved_frames(FD, fi2, fps2, OUT_TEMP, Encoder, Quality, INPUT_PATH)

            # Clean up resized temp
            try: os.remove(temp_resized)
            except: pass

            # Remux audio from original input
            remux_audio(OUT_TEMP, OUT, INPUT_PATH)
            if not os.path.exists(OUT) and os.path.exists(OUT_TEMP):
                shutil.move(OUT_TEMP, OUT)
                print(f"  (Moved temp → final: remux failed, using temp)")
            try: os.remove(OUT_TEMP)
            except: pass

            fi, fps, tf, stopped = fi2, fps2, tf2, stopped2

    td = fi
    if stopped:
        if Verbose_Log == "ON":
            print(f"  Partial output: {td} frames (stopped by user)")
    vd = td / fps if fps > 0 else 0

    if Verbose_Log == "ON":
        print(f"\n  ✓ Output: {OUT}")
        print(f"  Frames: {td}, Duration: {vd:.1f}s")
        print(f"  Encoder: {Encoder}, Quality: {Quality}")
        print(f"  Engine: TensorRT {_PRECISION_TAG.upper()} ({SCALE}x)")
        print(f"  Pipeline: TensorRT {_PRECISION_TAG.upper()} ({'2-pass' if PASS_COUNT == 2 else 'single-pass'})")

    if os.path.exists(OUT):
        out_size = os.path.getsize(OUT) / (1024*1024)
        print(f"  Output File Size: {out_size:.1f} MB")
        print(f"  Output Resolution: {out_w}x{out_h} ({detect_orientation(out_w, out_h)})")
        print(f"  Output FPS: {fps:.2f}")
    else:
        print(f"  ⚠ Output file not found: {OUT}")

    if Auto_Download == "ON":
        try:
            from google.colab import files
            files.download(OUT)
            print(f"  ✓ Auto download started")
        except:
            print(f"  (Auto download not available - file at {OUT})")
    else:
        print(f"  (Auto download disabled - file at {OUT})")

print("\n  ✓ Done.")

In [ ]:
#@title Step 2 - Worker (leave running)
# AI Upscaler - Colab worker. Polls a Google Drive folder for jobs written by the
# Android app, runs the ORIGINAL Proteus V3 Cell 2 with injected parameters,
# and writes status/progress/output back to Drive. No tunnels, no public server.
import os, re, sys, json, time, shutil, signal, subprocess, threading, traceback

WORK   = "/content/work"
CELL2  = "/content/cell2_original.py"
WORKER_VERSION = 2
FOLDER = "application/vnd.google-apps.folder"
os.makedirs(WORK, exist_ok=True)

import io, threading as _th
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaIoBaseUpload, MediaFileUpload

_tl = _th.local()
def svc():  # httplib2 is not thread-safe -> one client per thread
    if not hasattr(_tl, "s"): _tl.s = build("drive", "v3", cache_discovery=False)
    return _tl.s

def find(parent, name, mime=None):
    q = f"'{parent}' in parents and name='{name}' and trashed=false"
    if mime: q += f" and mimeType='{mime}'"
    r = svc().files().list(q=q, fields="files(id,name)", pageSize=5).execute()
    return r["files"][0]["id"] if r["files"] else None

def ensure_folder(parent, name):
    fid = find(parent, name, FOLDER)
    if fid: return fid
    return svc().files().create(body={"name": name, "mimeType": FOLDER, "parents": [parent]}, fields="id").execute()["id"]

_ids = {}
def write_json(parent, name, obj):
    media = MediaIoBaseUpload(io.BytesIO(json.dumps(obj).encode()), mimetype="application/json")
    key = (parent, name); fid = _ids.get(key) or find(parent, name)
    if fid:
        svc().files().update(fileId=fid, media_body=media).execute()
    else:
        fid = svc().files().create(body={"name": name, "parents": [parent]}, media_body=media, fields="id").execute()["id"]
    _ids[key] = fid

def read_json(fid):
    return json.loads(svc().files().get_media(fileId=fid).execute())

def download(fid, path):
    with open(path, "wb") as f:
        d = MediaIoBaseDownload(f, svc().files().get_media(fileId=fid), chunksize=32 * 1024 * 1024)
        done = False
        while not done: _, done = d.next_chunk()

def upload(path, parent, name):
    media = MediaFileUpload(path, resumable=True, chunksize=32 * 1024 * 1024)
    req = svc().files().create(body={"name": name, "parents": [parent]}, media_body=media, fields="id,size")
    resp = None
    while resp is None: _, resp = req.next_chunk(num_retries=5)
    return int(resp["size"])

ROOT_ID = JOBS_ID = None

ALLOWED = {
  "scale": {1, 2},
  "preset": {"Custom", "V1 Anime 1 Pass", "V2 Anime 2 Pass", "V3 Anime High Quality 2 Pass"},
  "input_res": {"1080p", "1440p", "2160p", "No Resize"},
  "output_res": {"Original AI Output", "1080p HD", "1440p 2K", "2160p 4K"},
  "encoder": {"x264 8bit", "x265 8bit", "x265 10bit", "ProRes 4444 12bit", "FFV1 16bit Lossless"},
}
SLIDERS = {  # json key -> (cell2 variable, min, max)
  "anti_alias_deblur": ("Anti_Alias_Deblur", -100, 100), "reduce_noise": ("ReduceNoise", 0, 100),
  "recover_details": ("RecoverDetails", 0, 100), "dehalo": ("Dehalo", 0, 100),
  "sharpen": ("Sharpen", 0, 100), "revert_compression": ("RevertCompression", 0, 100),
  "recover_original": ("Recover_Original_Details", 0, 100)}

def gpu_name():
    try: return subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip()
    except Exception: return None

_state = {"state": "starting"}
def heartbeat_loop():
    while True:
        try: write_json(ROOT_ID, "backend.json", {**_state, "ts": time.time(), "gpu": gpu_name(), "worker_version": WORKER_VERSION})
        except Exception as e: print("heartbeat error:", e)
        time.sleep(5)

def validate(job):
    s = job.get("settings", {})
    def pick(k):
        v = s.get(k)
        if v not in ALLOWED[k]: raise ValueError(f"invalid {k}: {v!r}")
        return v
    out = {k: pick(k) for k in ALLOWED}
    out["fp16"] = bool(s.get("fp16", True))
    q = int(s.get("quality", 16))
    if not 0 <= q <= 51: raise ValueError("quality out of range")
    out["quality"] = q
    sl = s.get("sliders", {})
    out["sliders"] = {}
    for k, (_, lo, hi) in SLIDERS.items():
        v = int(sl.get(k, 0))
        if not lo <= v <= hi: raise ValueError(f"{k} out of range")
        out["sliders"][k] = v
    return out

def build_script(s, input_path, dst):
    """Original Cell 2 with only its `#@param` form lines replaced."""
    vals = {"INPUT_PATH": input_path, "SCALE": f"{s['scale']}x", "Quality_Preset": s["preset"],
            "Upscaling_Speed_Up": "ON" if s["fp16"] else "OFF",
            "Input_Video_Resolution": s["input_res"], "Input_Image_Resolution": "1440p",
            "Output_Video_Resize": s["output_res"], "Image_Format": "PNG 8bit",
            "Encoder": s["encoder"], "Save_Frames": "OFF", "Quality": s["quality"],
            "Auto_Download": "OFF"}
    for k, (var, _, _) in SLIDERS.items(): vals[var] = s["sliders"][k]
    src = open(CELL2, encoding="utf-8").read().replace("\r", "")
    done = set()
    def sub(m):
        name = m.group(1)
        if name not in vals: return m.group(0)
        done.add(name); return f"{name} = {vals[name]!r}"
    src = re.sub(r"^([A-Za-z_]+) = .*#@param.*$", sub, src, flags=re.M)
    missing = set(vals) - done
    if missing: raise RuntimeError(f"Cell 2 layout changed, params not injected: {missing}")
    open(dst, "w", encoding="utf-8").write(src)

def probe(path):
    r = subprocess.run(["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries",
        "stream=width,height,r_frame_rate,nb_frames:format=duration", "-of", "json", path],
        capture_output=True, text=True, timeout=60)
    d = json.loads(r.stdout); st = d["streams"][0]
    n, m = st["r_frame_rate"].split("/"); fps = float(n) / float(m or 1)
    frames = int(st["nb_frames"]) if str(st.get("nb_frames", "")).isdigit() else int(float(d["format"]["duration"]) * fps)
    return int(st["width"]), int(st["height"]), fps, max(frames, 1)

CANCEL = threading.Event()

def run_job(jid, jd):
    wd = f"{WORK}/{jid}"; log = []
    def st(**kw):
        try: write_json(jd, "status.json", {"job": jid, "ts": time.time(), **kw})
        except Exception as e: print("status write failed:", e)
    try:
        jf = find(jd, "job.json")
        if not jf: raise FileNotFoundError("job.json missing")
        job = read_json(jf); s = validate(job)
        src = find(jd, job["input_file"])
        if not src: raise FileNotFoundError("input missing")
        os.makedirs(wd, exist_ok=True)
        st(state="preparing", message="Copying video to GPU runtime")
        ext = os.path.splitext(job["input_file"])[1] or ".mp4"
        local_in = f"{wd}/in{ext}"; download(src, local_in)
        if os.path.getsize(local_in) != job["input_size"]: raise IOError("input incomplete")
        w, h, fps, total = probe(local_in)
        passes = 2 if s["preset"] in ("V2 Anime 2 Pass", "V3 Anime High Quality 2 Pass") else 1
        script = f"{wd}/run.py"; build_script(s, local_in, script)
        st(state="processing", progress=0.0, message="Starting AI engine", in_w=w, in_h=h, fps=fps, frames=total)
        p = subprocess.Popen([sys.executable, script], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             cwd=wd, env=dict(os.environ, PYTHONUNBUFFERED="1"), start_new_session=True)
        buf, cur_pass, last_write, last_cancel, prog = b"", 1, 0, 0, 0.0
        while True:
            if time.time() - last_cancel > 4:
                last_cancel = time.time()
                if find(jd, "cancel"):
                    os.killpg(os.getpgid(p.pid), signal.SIGTERM); p.wait()
                    st(state="cancelled"); return
            ch = p.stdout.read1(4096)
            if not ch and p.poll() is not None: break
            buf += ch
            parts = re.split(rb"[\r\n]", buf); buf = parts.pop()
            for ln in parts:
                t = ln.decode("utf-8", "ignore").strip()
                if t: log.append(t); log = log[-300:]
                m = re.search(r"(Pass 1|Pass 2|Processing) Frame\s+(\d+)", t)
                if m:
                    cur_pass = 2 if m.group(1) == "Pass 2" else 1
                    prog = min(((cur_pass - 1) + int(m.group(2)) / total) / passes, 0.99)
            if time.time() - last_write > 3:
                st(state="processing", progress=round(prog, 4), pass_no=cur_pass, passes=passes,
                   message="Upscaling" + (f" (pass {cur_pass}/{passes})" if passes > 1 else ""))
                last_write = time.time()
        if p.returncode != 0: raise RuntimeError("engine exited with code %s" % p.returncode)
        outs = [f for f in os.listdir(wd) if f.startswith("KZM.FX.") and "_temp" not in f and "_pass1" not in f]
        if not outs: raise RuntimeError("no output produced")
        out = outs[0]
        st(state="finalizing", progress=0.99, message="Sending result to Drive")
        size = upload(f"{wd}/{out}", jd, out)
        st(state="done", progress=1.0, output_file=out, output_size=size)
    except Exception as e:
        st(state="failed", error=str(e)[:300], log_tail="\n".join(log[-40:]) + "\n" + traceback.format_exc()[-1500:])
    finally:
        shutil.rmtree(wd, ignore_errors=True)

def main():
    global ROOT_ID, JOBS_ID
    ROOT_ID = ensure_folder("root", "AI Upscaler"); JOBS_ID = ensure_folder(ROOT_ID, "jobs")
    threading.Thread(target=heartbeat_loop, daemon=True).start()
    _state["state"] = "ready"; print("Worker ready. Waiting for jobs.")
    seen = set()
    while True:
        try:
            r = svc().files().list(q=f"'{JOBS_ID}' in parents and mimeType='{FOLDER}' and trashed=false",
                                   fields="files(id,name)", pageSize=100).execute()
            for f in sorted(r["files"], key=lambda x: x["name"]):
                if f["id"] in seen: continue
                if not find(f["id"], "job.json") or find(f["id"], "status.json"): continue
                seen.add(f["id"]); _state.update(state="busy", job=f["name"])
                print("Job", f["name"]); run_job(f["name"], f["id"]); _state.update(state="ready", job=None)
        except Exception as e:
            print("poll error:", e)
        time.sleep(4)

if __name__ == "__main__": main()

